In [ ]:
import os, re, gc, random, json, unicodedata, warnings, subprocess, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import transformers

print("=" * 60)
print("  Environment")
print("=" * 60)
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM total   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Install CORAL ordinal loss (one-time per Kaggle session)
result = subprocess.run(["pip", "install", "coral-pytorch", "-q"],
                        capture_output=True, text=True)
print(result.stdout); print(result.stderr); print("✅ coral-pytorch installed")

from coral_pytorch.losses import corn_loss
from coral_pytorch.dataset import corn_label_from_logits
from coral_pytorch.layers import CoralLayer
print("✅ CORAL imported")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
class Config:
    """Single source of truth for all hyperparameters.
    
    Keep this T4-safe: batch 16, seq 256, adapter bottleneck 64.
    Total adapter overhead is ~1.2 M params — well within VRAM.
    """
    # ── Backbone
    MODEL_NAME       = "xlm-roberta-base"   # 278 M params, 12 layers, 768-d
    
    # ── Tokeniser
    MAX_LEN          = 256                  # covers ~95% of posts
    
    # ── CharCNN (shared by M1 + M2)
    MAX_CHAR         = 20                   # chars per subword token
    CHAR_EMBED_DIM   = 32                   # character embedding dim
    CHAR_FILTERS     = (2, 3, 4)            # bigram/trigram/tetragram convs
    CHAR_NUM_FILTERS = 64                   # filters per width → out = 192
    CHAR_DROPOUT     = 0.3
    
    # ── M2: Character-Gated Adapter (NEW)
    ADAPTER_SIZE     = 64                   # bottleneck dim (768 → 64 → 768)
    N_ENCODER_LAYERS = 12                   # XLM-R-base has 12 layers
    
    # ── Training
    BATCH_SIZE       = 16
    EPOCHS           = 15
    PATIENCE         = 5
    XLMR_LR          = 2e-5                 # pretrained backbone (slow)
    CHAR_LR          = 4e-5                 # CharCNN + char_proj
    ADAPTER_LR       = 5e-5                 # M2 adapters (new, train from 0)
    HEAD_LR          = 5e-5                 # CORAL head
    WEIGHT_DECAY     = 0.05
    CLIP_GRAD        = 1.0
    
    # ── Labels
    NUM_CLASSES      = 3
    LABEL_MAP        = {"mild": 0, "moderate": 1, "severe": 2}
    IDX2LABEL        = {0: "mild", 1: "moderate", 2: "severe"}
    
    # ── Paths (adjust to your Kaggle dataset)
    _BASE = "/kaggle/input/datasets/ouayresoumaima/data-char-drj"
    TRAIN_PATH = f"{_BASE}/darija_train.csv"
    VAL_PATH   = f"{_BASE}/darija_val.csv"
    TEST_PATH  = f"{_BASE}/darija_test.csv"      # gold balanced 78-post test
    SAVE_DIR   = "/kaggle/working"
    
    # ── Reproducibility
    SEED   = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = Config.SEED) -> None:
    """Pin every RNG for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


seed_everything()
os.makedirs(Config.SAVE_DIR, exist_ok=True)

print("=" * 60)
print("  Configuration")
print("=" * 60)
print(f"  Device       : {Config.DEVICE}")
print(f"  Backbone     : {Config.MODEL_NAME}")
print(f"  Seq length   : {Config.MAX_LEN} tokens")
print(f"  Chars/token  : {Config.MAX_CHAR}")
print(f"  CharCNN out  : {len(Config.CHAR_FILTERS)*Config.CHAR_NUM_FILTERS}-d")
print(f"  Adapter dim  : 768 → {Config.ADAPTER_SIZE} → 768  (× 12 layers)")
print(f"  Batch size   : {Config.BATCH_SIZE}")
print(f"  Max epochs   : {Config.EPOCHS}  (patience={Config.PATIENCE})")
print(f"  LR backbone  : {Config.XLMR_LR}")
print(f"  LR CharCNN   : {Config.CHAR_LR}")
print(f"  LR adapter   : {Config.ADAPTER_LR}")
print(f"  LR head      : {Config.HEAD_LR}")

In [ ]:
def _load_split(primary: str, fallbacks: list = None) -> pd.DataFrame:
    """Try primary path, then fallbacks. Ensures 'label' (int) and 'text' (str)."""
    paths = [primary] + (fallbacks or [])
    for p in paths:
        if os.path.exists(p):
            df = pd.read_csv(p)
            if "label" not in df.columns and "weak_label" in df.columns:
                df["label"] = df["weak_label"].map(Config.LABEL_MAP)
            df["label"] = df["label"].astype(int)
            df["text"]  = df["text"].fillna("").astype(str)
            return df
    raise FileNotFoundError(f"No file found at any of: {paths}")


_fb = "/kaggle/working"
train_df = _load_split(Config.TRAIN_PATH, [f"{_fb}/darija_train.csv"])
val_df   = _load_split(Config.VAL_PATH,   [f"{_fb}/darija_val.csv"])
test_df  = _load_split(Config.TEST_PATH,  [f"{_fb}/darija_test.csv"])

# ── Integrity audit
print("=" * 62)
print("  DATA INTEGRITY AUDIT")
print("=" * 62)
issues = []
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    if df["label"].isna().sum(): issues.append(f"[{name}] NaN labels")
    if df["text"].isna().sum():  issues.append(f"[{name}] NaN texts")
    if (~df["label"].isin([0,1,2])).sum(): issues.append(f"[{name}] bad labels")
    if df.duplicated(subset="text").sum(): issues.append(f"[{name}] dup texts")

tr_te = set(train_df["text"]) & set(test_df["text"])
vl_te = set(val_df["text"])   & set(test_df["text"])
if tr_te: issues.append(f"LEAK train↔test : {len(tr_te)}")
if vl_te: issues.append(f"LEAK val↔test   : {len(vl_te)}")

print(f"\n  {'Split':8s}  {'N':>6}  {'mild':>6}  {'mod':>6}  {'sev':>6}")
print("  " + "─" * 42)
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name:8s}  {len(df):>6}  "
          f"{(df['label']==0).sum():>6}  "
          f"{(df['label']==1).sum():>6}  "
          f"{(df['label']==2).sum():>6}")

print(f"\n  Overlap  train↔test: {len(tr_te)}   val↔test: {len(vl_te)}")
if issues:
    print("\n  ❌ ISSUES:")
    for e in issues: print(f"    → {e}")
    raise RuntimeError("Data integrity check failed.")
print("  ✓ All integrity checks passed")
print("=" * 62)

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Light-touch normalization. Arabizi digits {2,3,5,7,9} are KEPT —
# they are the CharCNN's target signal. Casing is KEPT — XLM-R is
# case-sensitive.
# ─────────────────────────────────────────────────────────────────────
_RE_DIACRITICS = re.compile(r'[ؐ-ًؚ-ٰٟۖ-ۜ۟-۪ۤۧۨ-ۭ]')
_RE_TATWEEL    = re.compile(r'ـ+')
_RE_ALEF       = re.compile(r'[أإآٱ]')
_RE_YEH_ALT    = re.compile(r'ى')
_RE_URL        = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
_RE_MENTION    = re.compile(r'\bu/\w+')
_RE_SUB        = re.compile(r'\br/\w+')
_RE_ZERO_WIDTH = re.compile(r'[\u200b-\u200d\u2060\ufeff\u00ad]')

_HTML_MAP = [("&amp;","&"), ("&lt;","<"), ("&gt;",">"), ("&quot;",'"'),
             ("&#39;","'"), ("&nbsp;"," "), ("&#x200B;",""),
             ("&thinsp;"," "), ("&mdash;","—"), ("&ndash;","–"),
             ("&hellip;","…")]


def preprocess(text: str) -> str:
    """Idempotent normalization for Darija Reddit posts."""
    if not isinstance(text, str): return ""
    text = unicodedata.normalize("NFC", text)
    for old, new in _HTML_MAP: text = text.replace(old, new)
    text = _RE_ZERO_WIDTH.sub("", text)
    text = _RE_MENTION.sub("",  text)
    text = _RE_SUB.sub("",      text)
    text = _RE_URL.sub("",      text)
    text = _RE_DIACRITICS.sub("", text)
    text = _RE_TATWEEL.sub("",    text)
    text = _RE_ALEF.sub("ا",   text)
    text = _RE_YEH_ALT.sub("ي", text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ',   text)
    return text.strip()


print("=" * 62)
print("  PREPROCESSING")
print("=" * 62)
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    original   = df["text"].copy()
    df["text"] = df["text"].apply(preprocess)
    changed    = (df["text"] != original).sum()
    print(f"  [{name}] modified {changed:,}/{len(df):,} "
          f"({100*changed/len(df):.1f}%)")

# Drop near-empty rows
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    before = len(df)
    df.drop(df[df["text"].str.len() < 15].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    if before - len(df):
        print(f"  [{name}] dropped {before-len(df)} rows < 15 chars")

# Save preprocessed copies (so loader fallback works)
train_df.to_csv(f"{Config.SAVE_DIR}/darija_train.csv", index=False)
val_df.to_csv(  f"{Config.SAVE_DIR}/darija_val.csv",   index=False)
test_df.to_csv( f"{Config.SAVE_DIR}/darija_test.csv",  index=False)
print(f"  ✓ Saved to {Config.SAVE_DIR}/")

In [ ]:
def build_char_vocab(dfs: list) -> dict:
    """Build char vocab from train+val+test. Guarantees Arabizi digits."""
    chars = set()
    for df in dfs:
        for text in df["text"]: chars.update(str(text))
    chars.update("23579")  # Arabizi digit-letters guaranteed
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for i, c in enumerate(sorted(chars), start=2): vocab[c] = i
    return vocab


char_vocab = build_char_vocab([train_df, val_df, test_df])

print("=" * 58)
print(f"  CHAR VOCAB SIZE : {len(char_vocab)}")
print("=" * 58)
print("  Arabizi digit-letters:")
for d, ar, ex in [("2","hamza","m2assif"), ("3","ayn","3yit"),
                   ("5","kha","5oya"),    ("7","ha","7zen"),
                   ("9","qaf","9lbi")]:
    print(f"    '{d}'  idx={char_vocab[d]:>3d}   ({ar})   e.g. {ex}")

In [ ]:
class DarijaDataset(Dataset):
    """Pre-tokenises and pre-computes char_ids at init for speed."""
    _SPECIAL = {"<s>", "</s>", "<pad>", "<unk>"}

    def __init__(self, df, tokenizer, char_vocab,
                 max_len=Config.MAX_LEN, max_char=Config.MAX_CHAR):
        self.tok, self.cv = tokenizer, char_vocab
        self.max_len, self.max_char = max_len, max_char
        self.labels = df["label"].astype(int).tolist()
        texts = df["text"].fillna("").astype(str).tolist()
        self._input_ids, self._attention_mask, self._char_ids = [], [], []
        print(f"  Pre-computing char_ids for {len(texts):,} samples…",
              flush=True)
        for text in texts:
            enc = tokenizer(text, max_length=max_len, padding="max_length",
                            truncation=True, return_tensors="pt")
            ids  = enc["input_ids"].squeeze(0)
            mask = enc["attention_mask"].squeeze(0)
            tokens   = tokenizer.convert_ids_to_tokens(ids.tolist())
            char_mat = [self._token_chars(t) for t in tokens]
            self._input_ids.append(ids)
            self._attention_mask.append(mask)
            self._char_ids.append(torch.tensor(char_mat, dtype=torch.long))

    def _token_chars(self, token_str: str) -> list:
        if token_str in self._SPECIAL: return [0] * self.max_char
        clean = token_str.lstrip("▁")
        ids   = [self.cv.get(c, 1) for c in clean[:self.max_char]]
        ids  += [0] * (self.max_char - len(ids))
        return ids

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self._input_ids[idx],
            "attention_mask": self._attention_mask[idx],
            "char_ids":       self._char_ids[idx],
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

In [ ]:
# Build tokenizer + datasets + loaders + class weights
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

train_ds = DarijaDataset(train_df, tokenizer, char_vocab)
val_ds   = DarijaDataset(val_df,   tokenizer, char_vocab)
test_ds  = DarijaDataset(test_df,  tokenizer, char_vocab)

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE*2,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config.BATCH_SIZE*2,
                          shuffle=False, num_workers=2, pin_memory=True)

_cw = compute_class_weight("balanced", classes=np.array([0,1,2]),
                            y=train_df["label"].values)
class_weights = torch.tensor(_cw, dtype=torch.float).to(Config.DEVICE)

_b = next(iter(train_loader))
print("=" * 58)
print("  Datasets & Loaders")
print("=" * 58)
print(f"  Train : {len(train_ds):,}  →  {len(train_loader)} batches")
print(f"  Val   : {len(val_ds):,}  →  {len(val_loader)} batches")
print(f"  Test  : {len(test_ds):,}  →  {len(test_loader)} batches")
print(f"\n  Batch shapes:")
for k, v in _b.items(): print(f"    {k:16s}: {tuple(v.shape)}")
print(f"\n  Class weights: mild={_cw[0]:.3f} mod={_cw[1]:.3f} sev={_cw[2]:.3f}")

In [ ]:
class CharCNN(nn.Module):
    """
    Character-level CNN producing one feature vector per subword token.
    
    Used in:
        M1 — projected to 768 and added to subword embeddings
        M2 — used (in raw 192-d form) as the conditioning signal for gates
    
    Filter widths 2/3/4 capture Arabizi n-grams:
        '3y' (عي), '7z' (حز), 'bka', 'nmout', etc.
    Max-over-time pooling → length-invariant 192-d output per token.
    """
    def __init__(self, vocab_size,
                 char_embed_dim=Config.CHAR_EMBED_DIM,
                 filter_sizes=Config.CHAR_FILTERS,
                 num_filters=Config.CHAR_NUM_FILTERS,
                 dropout=Config.CHAR_DROPOUT):
        super().__init__()
        self.char_embed = nn.Embedding(vocab_size, char_embed_dim,
                                        padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_embed_dim, num_filters, fs, padding=fs//2)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.out_dim = num_filters * len(filter_sizes)  # 192

    def forward(self, char_ids):
        # char_ids: [B, T, C]
        B, T, C = char_ids.shape
        x = self.char_embed(char_ids.view(B*T, C))     # [B*T, C, embed]
        x = x.transpose(1, 2)                           # [B*T, embed, C]
        pooled = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        out = self.dropout(torch.cat(pooled, dim=-1))   # [B*T, 192]
        return out.view(B, T, -1)                       # [B, T, 192]

In [ ]:
class BaselineXLMR(nn.Module):
    """Standard XLM-R fine-tune with CORAL ordinal head. No char info."""
    def __init__(self, model_name=Config.MODEL_NAME,
                 num_classes=Config.NUM_CLASSES, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.xlmr(input_ids=input_ids,
                        attention_mask=attention_mask
                       ).last_hidden_state[:, 0]
        return self.classifier(cls)

In [ ]:
class DarijaCharCNN(nn.Module):
    """
    M1 — CharCNN features added (residually) to XLM-R subword embeddings
    BEFORE the encoder stack. The character signal enters at the input;
    all 12 encoder layers see enriched embeddings.
    """
    def __init__(self, model_name=Config.MODEL_NAME,
                 char_vocab_size=None,
                 num_classes=Config.NUM_CLASSES, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        _vs = char_vocab_size or len(char_vocab)
        self.char_cnn  = CharCNN(_vs)
        self.char_proj = nn.Linear(self.char_cnn.out_dim, d)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            CoralLayer(d, num_classes),
        )
        nn.init.normal_(self.char_proj.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.char_proj.bias)

        # Staged training
        self._char_active = False
        self._set_char_grad(False)
        self._char_feat = None

        def _emb_hook(module, input, output):
            if self._char_feat is not None and self._char_active:
                return output + self._char_feat
            return output
        self.xlmr.embeddings.word_embeddings.register_forward_hook(_emb_hook)

    def _set_char_grad(self, requires_grad):
        for p in self.char_cnn.parameters():  p.requires_grad = requires_grad
        for p in self.char_proj.parameters(): p.requires_grad = requires_grad

    def forward(self, input_ids, attention_mask, char_ids, **kwargs):
        self._char_feat = (self.char_proj(self.char_cnn(char_ids))
                            if self._char_active else None)
        outputs = self.xlmr(input_ids=input_ids,
                            attention_mask=attention_mask)
        self._char_feat = None
        return self.classifier(outputs.last_hidden_state[:, 0])

In [ ]:
class CharGatedAdapter(nn.Module):
    """
    One character-gated bottleneck adapter — placed INSIDE one encoder layer.
    
    Equation:
        adapter(h)  = W_up · GELU(W_down · h)            # 768 → 64 → 768
        gate        = sigmoid(W_gate · [h ; c])           # token-wise scalar
        h'          = h + gate · adapter(h)               # residual
    
    Where:
        h: [B, T, 768]  hidden states from the preceding sublayer
        c: [B, T, 192]  per-token character features from CharCNN (fixed
                         per forward pass — same c is fed to all 12 adapters)
    
    Init strategy:
        - W_up initialised to N(0, 1e-3²)  → adapter ≈ 0 at start
        - W_gate, b_gate initialised to 0  → gate = sigmoid(0) = 0.5
        Result: training begins at the M0 manifold and *learns* to deviate.
        
        Math note: W_gate = 0 means gate has no c-dependence on step 1,
        so CharCNN receives zero gradient on the very first optimizer step.
        After step 1, W_gate becomes non-zero and CharCNN trains normally.
        This is intentional — it guarantees a perfect M0-equivalent start.
    
    Parameter count per adapter:
        down (768·64 + 64)       =  49,216
        up   (64·768 + 768)      =  50,176
        gate ((768+192)·1 + 1)   =     961
        ─────────────────────────────────
        Total                    = 100,353  per adapter
        × 12 layers              = 1,204,236  (~0.44 % of XLM-R-base)
    """
    def __init__(self, hidden_size=768,
                 bottleneck=Config.ADAPTER_SIZE,
                 char_dim=192):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.up   = nn.Linear(bottleneck, hidden_size)
        self.gate = nn.Linear(hidden_size + char_dim, 1)

        # Near-zero init → adapter starts as identity, gate starts at 0.5
        nn.init.normal_(self.up.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.up.bias)
        nn.init.zeros_(self.gate.weight)
        nn.init.zeros_(self.gate.bias)

    def forward(self, h: torch.Tensor, c: torch.Tensor):
        """
        Args:
            h: [B, T, hidden]   hidden states
            c: [B, T, char_dim] character features (per-token, fixed)
        Returns:
            h_new: [B, T, hidden]  adapted hidden states
            gate : [B, T]          gate values (for analysis)
        """
        adapter_out = self.up(F.gelu(self.down(h)))           # [B, T, 768]
        gate_input  = torch.cat([h, c], dim=-1)               # [B, T, 960]
        gate        = torch.sigmoid(self.gate(gate_input))    # [B, T, 1]
        h_new       = h + gate * adapter_out
        return h_new, gate.squeeze(-1)

In [ ]:
class CharGatedAdapterXLMR(nn.Module):
    """
    M2 — XLM-RoBERTa with Character-Gated Adapters inside every encoder
    layer.
    
    Architecture (the contribution):
    
        Token IDs
            ↓
        XLM-R Embeddings (UNMODIFIED)
            ↓
            ├──────────────────┐
            ↓                  ↓
        Layer 1 ─ adapter+gate ←  c = CharCNN(char_ids)  [B,T,192]
            ↓                  ↑
        Layer 2 ─ adapter+gate ←┤
            ↓                  ↑
           ...                 │
            ↓                  ↑
        Layer 12 ─ adapter+gate┘
            ↓
        CLS → Dropout → CORAL → logits
    
    The CharCNN output is computed ONCE per forward pass and broadcast
    to all 12 adapters via internal buffers. Each adapter has its own
    down/up/gate parameters.
    
    Note on M1 vs M2:
        M1 injects characters at the EMBEDDING layer (one location, before
        the encoder). M2 injects characters INSIDE each of the 12 encoder
        layers, with token-wise gating. M2 is therefore a strict encoder-
        internal modification, addressing the criticism that M1 was only
        a pre-encoder modification.
    """
    def __init__(self, model_name=Config.MODEL_NAME,
                 char_vocab_size=None,
                 num_classes=Config.NUM_CLASSES,
                 adapter_size=Config.ADAPTER_SIZE, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size              # 768
        n_layers = self.xlmr.config.num_hidden_layers # 12

        _vs = char_vocab_size or len(char_vocab)
        self.char_cnn = CharCNN(_vs)
        char_dim = self.char_cnn.out_dim              # 192

        # One adapter per encoder layer
        self.adapters = nn.ModuleList([
            CharGatedAdapter(hidden_size=d,
                             bottleneck=adapter_size,
                             char_dim=char_dim)
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )

        # ── Staging state
        self._char_active = False          # whether CharCNN + adapters fire
        self._set_extension_grad(False)
        self._char_feat = None             # [B, T, 192] populated per fwd
        self._track_gates = False          # set True to collect gate values
        self._gate_log    = []             # collected gate tensors (eval)

        # ── Register per-layer hooks
        self._hook_handles = []
        for layer_idx, layer in enumerate(self.xlmr.encoder.layer):
            h = layer.register_forward_hook(self._make_layer_hook(layer_idx))
            self._hook_handles.append(h)

    def _make_layer_hook(self, idx: int):
        """Factory: closes over idx so each layer gets its own adapter.
        
        IMPORTANT: XLM-R encoder layers always return a tuple
        (hidden_states, ...optional). We MUST preserve the tuple wrapper
        when returning, otherwise the next layer sees a bare tensor and
        crashes. The branch below handles both signatures defensively.
        """
        adapter = self.adapters[idx]
        def _hook(module, inputs, output):
            # If staging inactive OR no char features → pass through
            if not self._char_active or self._char_feat is None:
                return output
            if isinstance(output, tuple):
                h = output[0]
                h_new, gate = adapter(h, self._char_feat)
                if self._track_gates:
                    self._gate_log.append(gate.detach())
                # Always preserve the tuple signature (even if 1-element)
                return (h_new,) + output[1:]
            else:
                h_new, gate = adapter(output, self._char_feat)
                if self._track_gates:
                    self._gate_log.append(gate.detach())
                return h_new
        return _hook

    def _set_extension_grad(self, requires_grad: bool):
        """Toggle gradients for ALL added modules (CharCNN + adapters)."""
        for p in self.char_cnn.parameters():  p.requires_grad = requires_grad
        for p in self.adapters.parameters():  p.requires_grad = requires_grad

    def forward(self, input_ids, attention_mask, char_ids, **kwargs):
        # Reset gate logger
        if self._track_gates: self._gate_log = []
        # Compute char features ONCE; broadcast to all 12 hooks via attribute
        if self._char_active:
            self._char_feat = self.char_cnn(char_ids)   # [B, T, 192]
        else:
            self._char_feat = None
        outputs = self.xlmr(input_ids=input_ids,
                            attention_mask=attention_mask)
        self._char_feat = None
        cls = outputs.last_hidden_state[:, 0]            # [B, 768]
        return self.classifier(cls)

    # ── Convenience: parameter count & breakdown
    def param_breakdown(self) -> dict:
        backbone   = sum(p.numel() for p in self.xlmr.parameters())
        char       = sum(p.numel() for p in self.char_cnn.parameters())
        adapters   = sum(p.numel() for p in self.adapters.parameters())
        classifier = sum(p.numel() for p in self.classifier.parameters())
        total      = backbone + char + adapters + classifier
        return dict(backbone=backbone, char=char, adapters=adapters,
                    classifier=classifier, total=total,
                    overhead_pct=100*(char+adapters+classifier)/backbone)

In [ ]:
# Quick parameter-count sanity check (no GPU memory consumed)
_tmp = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
br = _tmp.param_breakdown()
print("=" * 58)
print("  M2 PARAMETER BREAKDOWN")
print("=" * 58)
print(f"  XLM-R backbone : {br['backbone']:>12,}  (frozen behaviour: train at 2e-5)")
print(f"  CharCNN        : {br['char']:>12,}  (LR 4e-5)")
print(f"  12 adapters    : {br['adapters']:>12,}  (LR 5e-5)  ← NEW")
print(f"  CORAL head     : {br['classifier']:>12,}  (LR 5e-5)")
print(f"  ───────────────────────────────────")
print(f"  TOTAL          : {br['total']:>12,}")
print(f"  Overhead vs M0 : {br['overhead_pct']:.2f}%")
del _tmp; gc.collect()

In [ ]:
def make_optimizer(model: nn.Module) -> AdamW:
    """Layer-grouped LR: backbone slow, new modules faster."""
    named = list(model.named_parameters())
    xlmr_p     = [p for n,p in named if n.startswith("xlmr.")]
    char_p     = [p for n,p in named if n.startswith("char_cnn.")
                                     or n.startswith("char_proj.")]
    adapter_p  = [p for n,p in named if n.startswith("adapters.")]
    head_p     = [p for n,p in named if n.startswith("classifier.")]
    groups = [{"params": xlmr_p, "lr": Config.XLMR_LR}]
    if char_p:    groups.append({"params": char_p,    "lr": Config.CHAR_LR})
    if adapter_p: groups.append({"params": adapter_p, "lr": Config.ADAPTER_LR})
    if head_p:    groups.append({"params": head_p,    "lr": Config.HEAD_LR})
    return AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


def compute_metrics(labels, preds, name="") -> dict:
    labels, preds = np.array(labels), np.array(preds)
    macro_f1  = f1_score(labels, preds, average="macro", zero_division=0)
    per_class = f1_score(labels, preds, average=None, zero_division=0,
                         labels=[0,1,2])
    qwk = cohen_kappa_score(labels, preds, weights="quadratic")
    mae = float(np.mean(np.abs(labels - preds)))
    if name:
        W = 54
        print(f"\n{'═'*W}\n  {name}\n{'═'*W}")
        print(f"  Macro-F1 : {macro_f1:.4f}")
        print(f"  QWK      : {qwk:.4f}   (ordinal-aware)")
        print(f"  MAE      : {mae:.4f}")
        print(f"  mild  F1 : {per_class[0]:.4f}")
        print(f"  mod   F1 : {per_class[1]:.4f}")
        print(f"  sev   F1 : {per_class[2]:.4f}")
        print(classification_report(labels, preds, zero_division=0, digits=4,
              target_names=["mild","moderate","severe"]))
    return dict(macro_f1=macro_f1, qwk=qwk, mae=mae,
                mild_f1=per_class[0], moderate_f1=per_class[1],
                severe_f1=per_class[2])


def _forward(model, batch):
    ids   = batch["input_ids"].to(Config.DEVICE)
    mask  = batch["attention_mask"].to(Config.DEVICE)
    chars = batch["char_ids"].to(Config.DEVICE)
    lbls  = batch["label"].to(Config.DEVICE)
    return model(input_ids=ids, attention_mask=mask, char_ids=chars), lbls


def _train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        loss = corn_loss(logits, lbls, num_classes=Config.NUM_CLASSES)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), Config.CLIP_GRAD)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return total_loss/len(loader), f1_score(lbls_all, preds_all,
                                            average="macro", zero_division=0)


@torch.no_grad()
def _eval_epoch(model, loader):
    model.eval()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        total_loss += corn_loss(logits, lbls,
                                num_classes=Config.NUM_CLASSES).item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return (total_loss/len(loader),
            f1_score(lbls_all, preds_all, average="macro", zero_division=0),
            preds_all, lbls_all)


def train_model(model, name: str, activate_at_epoch: int = 2):
    """
    Full training loop with linear-warmup scheduler and early stopping.
    
    Staged training: extension modules (CharCNN/adapters) are frozen for
    epoch 1 (only the head + backbone fine-tune), then activated at
    `activate_at_epoch`. This stabilises training when the new modules
    are randomly initialised.
    """
    model = model.to(Config.DEVICE)
    optimizer  = make_optimizer(model)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, total_steps//10, total_steps)

    best_f1, best_preds, best_labels = 0.0, None, None
    no_improve = 0
    ckpt = os.path.join(Config.SAVE_DIR, f"{name.replace(' ','_')}.pt")

    W = 70
    print(f"\n{'═'*W}\n  {name}")
    print(f"  Steps/epoch : {len(train_loader)}   "
          f"Total : {total_steps}   Warmup : {total_steps//10}")
    print(f"{'═'*W}")
    print(f"  {'Ep':>3}  {'TrLoss':>8}  {'TrF1':>7}  "
          f"{'VlLoss':>8}  {'VlF1':>7}  {'Best':>4}")
    print(f"  {'─'*56}")

    for epoch in range(1, Config.EPOCHS + 1):
        # Activate extension modules at the configured epoch
        if epoch == activate_at_epoch and hasattr(model, '_char_active'):
            if not model._char_active:
                model._char_active = True
                if hasattr(model, '_set_char_grad'):
                    model._set_char_grad(True)
                if hasattr(model, '_set_extension_grad'):
                    model._set_extension_grad(True)
                print(f"  ★ Extension modules ACTIVATED at epoch {epoch}")

        tr_loss, tr_f1 = _train_epoch(model, train_loader,
                                       optimizer, scheduler)
        vl_loss, vl_f1, preds, labels = _eval_epoch(model, val_loader)

        flag = ""
        if vl_f1 > best_f1:
            best_f1, best_preds, best_labels = vl_f1, preds, labels
            torch.save(model.state_dict(), ckpt)
            no_improve = 0; flag = " ✓"
        else:
            no_improve += 1

        print(f"  {epoch:>3}  {tr_loss:>8.4f}  {tr_f1:>7.4f}  "
              f"{vl_loss:>8.4f}  {vl_f1:>7.4f}{flag}")
        if no_improve >= Config.PATIENCE:
            print(f"\n  Early stop — best val F1 = {best_f1:.4f}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
    print(f"  Best checkpoint restored → val Macro-F1 = {best_f1:.4f}")
    return model, best_f1, best_preds, best_labels


print("✅ Training engine ready (CORAL + staged + early stop)")

In [ ]:
SEEDS = [42, 0, 1]   # canonical 3-seed set for reproducibility


def run_multi_seed(ModelClass: type, model_kwargs: dict, name: str):
    """Train ModelClass with each seed, return aggregated val+test metrics."""
    val_records, test_records, seed_checkpoints = [], [], {}

    for seed in SEEDS:
        seed_everything(seed)
        model = ModelClass(**model_kwargs)
        model, _, val_preds, val_labels = train_model(
            model, f"{name}_seed{seed}")
        val_records.append(compute_metrics(val_labels, val_preds))

        _, _, te_preds, te_labels = _eval_epoch(model, test_loader)
        test_records.append(compute_metrics(te_labels, te_preds))

        seed_checkpoints[seed] = os.path.join(
            Config.SAVE_DIR, f"{name}_seed{seed}.pt")

        del model
        gc.collect(); torch.cuda.empty_cache()

    def _agg(records):
        keys = records[0].keys()
        return {k: {"mean": float(np.mean([r[k] for r in records])),
                    "std":  float(np.std( [r[k] for r in records])),
                    "runs": [r[k] for r in records]} for k in keys}

    return {"val": _agg(val_records),
            "test": _agg(test_records),
            "checkpoints": seed_checkpoints}


print("✅ run_multi_seed ready")

In [ ]:
print("█"*70 + "\n  M0 — Baseline XLM-R + CORAL (3 seeds)\n" + "█"*70)
m0_results = run_multi_seed(BaselineXLMR, {}, "M0_Baseline")
with open(f"{Config.SAVE_DIR}/m0_results.json", "w") as f:
    json.dump({k: m0_results[k] for k in ["val","test"]}, f, indent=2)
print("✅ M0 done — saved to m0_results.json")                                                                          

In [ ]:
print("█"*70 + "\n  M1 — DarijaCharCNN (embedding injection, 3 seeds)\n" + "█"*70)
m1_results = run_multi_seed(
    DarijaCharCNN,
    {"char_vocab_size": len(char_vocab)},
    "M1_DarijaCharCNN")
with open(f"{Config.SAVE_DIR}/m1_results.json", "w") as f:
    json.dump({k: m1_results[k] for k in ["val","test"]}, f, indent=2)
print("✅ M1 done — saved to m1_results.json")

In [ ]:
def make_optimizer(model: nn.Module) -> AdamW:
    """Layer-grouped LR. Catches all extension modules cleanly."""
    named = list(model.named_parameters())
    xlmr_p    = [p for n,p in named if n.startswith("xlmr.")]
    char_p    = [p for n,p in named if n.startswith("char_cnn.")
                                     or n.startswith("char_proj.")]
    adapter_p = [p for n,p in named if n.startswith("adapters.")]
    head_p    = [p for n,p in named if n.startswith("classifier.")]
    # Sanity check: every parameter must be in exactly one group
    seen = set()
    for grp in [xlmr_p, char_p, adapter_p, head_p]:
        for p in grp: seen.add(id(p))
    all_p = {id(p) for n,p in named}
    missing = all_p - seen
    assert not missing, f"❌ {len(missing)} params not in any optimizer group"
    groups = [{"params": xlmr_p, "lr": Config.XLMR_LR}]
    if char_p:    groups.append({"params": char_p,    "lr": Config.CHAR_LR})
    if adapter_p: groups.append({"params": adapter_p, "lr": Config.ADAPTER_LR})
    if head_p:    groups.append({"params": head_p,    "lr": Config.HEAD_LR})
    print(f"  Optimizer groups: xlmr={len(xlmr_p)}, char={len(char_p)}, "
          f"adapter={len(adapter_p)}, head={len(head_p)}")
    return AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


print("✅ make_optimizer fixed — char_proj now in 'char' group")

In [ ]:
print("█"*70 + "\n  M2 — Character-Gated Adapter (encoder-internal, 3 seeds)\n" + "█"*70)
m2_results = run_multi_seed(
    CharGatedAdapterXLMR,
    {"char_vocab_size": len(char_vocab),
     "adapter_size": Config.ADAPTER_SIZE},
    "M2_CharGatedAdapter")
with open(f"{Config.SAVE_DIR}/m2_results.json", "w") as f:
    json.dump({k: m2_results[k] for k in ["val","test"]}, f, indent=2)
print("✅ M2 done — saved to m2_results.json")

In [ ]:
W = 100
METRICS = ["macro_f1", "qwk", "mae", "mild_f1", "moderate_f1", "severe_f1"]

def _fmt(d, k): return f"{d[k]['mean']:.4f}±{d[k]['std']:.4f}"

for split in ["val", "test"]:
    print("=" * W)
    print(f"  RESULTS — {split.upper()}  (mean ± std over {len(SEEDS)} seeds)")
    print("=" * W)
    print(f"  {'Metric':<13} | {'M0':>15} | {'M1':>15} | "
          f"{'M2 ★':>15} | {'Δ(M2-M0)':>11} | {'Δ(M2-M1)':>11}")
    print("  " + "─" * (W-2))
    for m in METRICS:
        m0 = m0_results[split]; m1 = m1_results[split]; m2 = m2_results[split]
        d20 = m2[m]['mean'] - m0[m]['mean']
        d21 = m2[m]['mean'] - m1[m]['mean']
        # For MAE, lower is better → flip sign of delta interpretation
        sign = "+" if (m != "mae") else "−"
        flag = ""
        # "real effect" rule: |Δ| > 2 × pooled std
        pooled = np.sqrt((m0[m]['std']**2 + m2[m]['std']**2)/2)
        if pooled > 0 and abs(d20) > 2*pooled: flag = " ✓"
        print(f"  {m:<13} | {_fmt(m0,m):>15} | {_fmt(m1,m):>15} | "
              f"{_fmt(m2,m):>15} | {d20:>+10.4f}{flag} | {d21:>+11.4f}")
    print("  " + "─" * (W-2))
    print("  ✓ = |Δ| > 2× pooled std (effect likely real, not seed noise)")
    print()

In [ ]:
def _arabizi_flag(df):
    if "has_arabizi_signal" in df.columns:
        return df["has_arabizi_signal"].astype(bool)
    return df["text"].str.contains(r"[23579]", regex=True)


arabizi_mask = _arabizi_flag(test_df).values


@torch.no_grad()
def _collect_preds(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        preds.extend(corn_label_from_logits(logits).cpu().tolist())
        labels.extend(lbls.cpu().tolist())
    return np.array(preds), np.array(labels)


# Use seed-42 models for the slice analysis (most reproducible)
print("=" * 70)
print("  ARABIZI SUBSET — seed 42 only")
print("  (Demonstrates that the encoder modification helps where it matters)")
print("=" * 70)

m0_m = BaselineXLMR().to(Config.DEVICE)
m0_m.load_state_dict(torch.load(m0_results["checkpoints"][42],
                                 map_location=Config.DEVICE))
m1_m = DarijaCharCNN(char_vocab_size=len(char_vocab)).to(Config.DEVICE)
m1_m.load_state_dict(torch.load(m1_results["checkpoints"][42],
                                 map_location=Config.DEVICE))
m2_m = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab)).to(Config.DEVICE)
m2_m.load_state_dict(torch.load(m2_results["checkpoints"][42],
                                 map_location=Config.DEVICE))
# Activate extensions (otherwise hooks are no-ops)
m1_m._char_active = True
m2_m._char_active = True

m0_p, true = _collect_preds(m0_m, test_loader)
m1_p, _    = _collect_preds(m1_m, test_loader)
m2_p, _    = _collect_preds(m2_m, test_loader)

for subset, mask in [
    ("All test posts",     np.ones(len(true), dtype=bool)),
    ("Arabizi posts",      arabizi_mask),
    ("Non-Arabizi posts",  ~arabizi_mask),
]:
    n = int(mask.sum())
    if n == 0:
        print(f"\n  [{subset}]  n=0 — skip"); continue
    f0 = f1_score(true[mask], m0_p[mask], average="macro", zero_division=0)
    f1 = f1_score(true[mask], m1_p[mask], average="macro", zero_division=0)
    f2 = f1_score(true[mask], m2_p[mask], average="macro", zero_division=0)
    print(f"\n  [{subset}]  n={n}")
    print(f"    M0 Macro-F1 : {f0:.4f}")
    print(f"    M1 Macro-F1 : {f1:.4f}   (Δ vs M0 : {f1-f0:+.4f})")
    print(f"    M2 Macro-F1 : {f2:.4f}   (Δ vs M0 : {f2-f0:+.4f}  ·  "
          f"Δ vs M1 : {f2-f1:+.4f}) ★")

In [ ]:
@torch.no_grad()
def collect_gate_statistics(model, loader, tokenizer, char_vocab):
    """
    Run the model on the test loader with gate tracking ON; compute
    mean gate value per layer for Arabizi-containing tokens vs plain tokens.
    """
    model.eval()
    model._track_gates = True
    arabizi_digit_ids = {char_vocab[d] for d in "23579" if d in char_vocab}

    # Per-layer accumulators
    n_layers = len(model.adapters)
    arabizi_sum   = torch.zeros(n_layers)
    arabizi_count = torch.zeros(n_layers)
    plain_sum     = torch.zeros(n_layers)
    plain_count   = torch.zeros(n_layers)

    for batch in loader:
        ids   = batch["input_ids"].to(Config.DEVICE)
        mask  = batch["attention_mask"].to(Config.DEVICE)
        chars = batch["char_ids"].to(Config.DEVICE)
        _ = model(input_ids=ids, attention_mask=mask, char_ids=chars)

        # Stack the 12 gate tensors
        gates = torch.stack(model._gate_log, dim=0)        # [12, B, T]
        # Build token-wise Arabizi mask: token contains any of {2,3,5,7,9}
        is_arabizi = torch.zeros_like(chars[..., 0], dtype=torch.bool)
        for did in arabizi_digit_ids:
            is_arabizi |= (chars == did).any(dim=-1)        # [B, T]
        # Only count real (non-pad) tokens
        real = mask.bool()                                  # [B, T]
        is_arabizi = is_arabizi & real
        is_plain   = real & ~is_arabizi

        for L in range(n_layers):
            g = gates[L].cpu()
            arabizi_sum[L]   += g[is_arabizi.cpu()].sum()
            arabizi_count[L] += is_arabizi.cpu().sum()
            plain_sum[L]     += g[is_plain.cpu()].sum()
            plain_count[L]   += is_plain.cpu().sum()

    model._track_gates = False
    model._gate_log    = []
    mean_arabizi = (arabizi_sum / arabizi_count.clamp(min=1)).numpy()
    mean_plain   = (plain_sum   / plain_count.clamp(min=1)).numpy()
    return mean_arabizi, mean_plain


print("=" * 70)
print("  GATE VALUE ANALYSIS — M2 seed 42")
print("  (Gate near 1.0 = adapter applied   ·   Gate near 0.0 = bypass)")
print("=" * 70)

gate_arabizi, gate_plain = collect_gate_statistics(
    m2_m, test_loader, tokenizer, char_vocab)

print(f"\n  {'Layer':>6} | {'Arabizi tokens':>15} | {'Plain tokens':>14} | {'Δ':>8}")
print("  " + "─" * 54)
for L in range(len(gate_arabizi)):
    d = gate_arabizi[L] - gate_plain[L]
    mark = " ★" if d > 0.02 else "  "
    print(f"  {L+1:>6} | {gate_arabizi[L]:>15.4f} | {gate_plain[L]:>14.4f} | {d:>+7.4f}{mark}")

print("\n  ★ = adapter opens significantly more on Arabizi tokens")
print("    (Empirical proof the encoder learned code-switching routing.)")

# Save for paper figure
np.save(f"{Config.SAVE_DIR}/gate_arabizi.npy", gate_arabizi)
np.save(f"{Config.SAVE_DIR}/gate_plain.npy",   gate_plain)
print(f"\n  Gate arrays saved for plotting in the paper.")

In [ ]:
final_report = {
    "config": {
        "model_name": Config.MODEL_NAME,
        "max_len": Config.MAX_LEN,
        "batch_size": Config.BATCH_SIZE,
        "epochs": Config.EPOCHS,
        "save_dir": Config.SAVE_DIR,
    }
}

print("=" * 70)
print(f"Results saved to {Config.SAVE_DIR}/")
print("    m0_results.json, m1_results.json, m2_results.json")
print("    final_report.json")
print("    gate_arabizi.npy, gate_plain.npy")
print("    M{0,1,2}_*_seed{42,0,1}.pt    (checkpoints)")
print("\n  → Paper §5 tables: read final_report.json['results']")
print("  → Paper figure   : plot gate_arabizi vs gate_plain")
print("=" * 70)

---
# § 11 · M2 — Character-Gated Adapter (Encoder-Internal Modification)

This section extends the prior work with a third model, **M2**, that addresses an architectural limitation of M1: while M1 injects character information at the embedding layer (before the encoder), the twelve transformer layers themselves remain unchanged from the pretrained backbone. M2 places a small character-gated adapter **inside every encoder layer**.

**Architecture.** Inside each of the 12 XLM-RoBERTa encoder layers, a bottleneck adapter (768 → 64 → 768) is inserted after the layer's standard sublayers. A token-wise sigmoid gate, conditioned on the concatenation of the hidden state and the CharCNN features, decides how strongly the adapter contributes:

$$\tilde{h}^{(\ell)} = h^{(\ell)} + g^{(\ell)} \odot \mathrm{Up}(\mathrm{GELU}(\mathrm{Down}(h^{(\ell)})))$$

where $g^{(\ell)} = \sigma(W_{\mathrm{gate}}^{(\ell)} [h^{(\ell)}; c])$ and $c \in \mathbb{R}^{T \times 192}$ is the per-token CharCNN output, shared across all twelve adapters.

**Parameter cost.** ~1.2M parameters (0.44% of XLM-R-base).

**Initialisation.** Up-projection initialised to $\mathcal{N}(0, 10^{-6})$, gate weights initialised to zero. At step 1 the adapter contribution is negligible (M2 ≈ M0); the model learns to deviate from this identity behaviour during training.

**Comparison protocol.** M0 and M1 results from §8 and §9 are reused without modification. M2 is trained under identical splits, seeds, preprocessing, tokenizer, and `train_model` engine. The only delta is the model class and the additional `adapters.*` parameter group in the optimizer.
---

In [ ]:
try:
    del model
except: pass
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── M2 building blocks ────────────────────────────────────────────────

class CharGatedAdapter(nn.Module):
    """One character-gated bottleneck adapter (placed inside one encoder layer).
    
    adapter(h) = W_up · GELU(W_down · h)            # 768 → 64 → 768
    gate       = sigmoid(W_gate · [h ; c])           # token-wise scalar
    h'         = h + gate * adapter(h)
    """
    def __init__(self, hidden_size=768, bottleneck=64, char_dim=192):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck)
        self.up   = nn.Linear(bottleneck, hidden_size)
        self.gate = nn.Linear(hidden_size + char_dim, 1)
        # Near-zero init: adapter ≈ 0 at start, gate = sigmoid(0) = 0.5
        nn.init.normal_(self.up.weight, mean=0.0, std=1e-3)
        nn.init.zeros_(self.up.bias)
        nn.init.zeros_(self.gate.weight)
        nn.init.zeros_(self.gate.bias)

    def forward(self, h, c):
        a = self.up(F.gelu(self.down(h)))
        g = torch.sigmoid(self.gate(torch.cat([h, c], dim=-1)))
        return h + g * a, g.squeeze(-1)


class CharGatedAdapterXLMR(nn.Module):
    """M2 — XLM-R with one CharGatedAdapter inside each of 12 encoder layers."""
    
    def __init__(self, model_name=Config.MODEL_NAME,
                 char_vocab_size=None,
                 num_classes=Config.NUM_CLASSES,
                 adapter_size=64):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        n_layers = self.xlmr.config.num_hidden_layers
        _vs = char_vocab_size or len(char_vocab)
        self.char_cnn = CharCNN(_vs)
        char_dim = self.char_cnn.out_dim  # 192
        
        self.adapters = nn.ModuleList([
            CharGatedAdapter(d, adapter_size, char_dim)
            for _ in range(n_layers)
        ])
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )
        
        # Staged training (compatible with existing train_model)
        self._char_active = False
        self._set_char_grad(False)
        self._char_feat = None
        self._track_gates = False
        self._gate_log = []
        
        # Register hooks on each encoder layer
        for idx, layer in enumerate(self.xlmr.encoder.layer):
            layer.register_forward_hook(self._make_hook(idx))

    def _make_hook(self, idx):
        adapter = self.adapters[idx]
        def _hook(module, inputs, output):
            if not self._char_active or self._char_feat is None:
                return output
            if isinstance(output, tuple):
                h_new, g = adapter(output[0], self._char_feat)
                if self._track_gates:
                    self._gate_log.append(g.detach())
                return (h_new,) + output[1:]
            h_new, g = adapter(output, self._char_feat)
            if self._track_gates:
                self._gate_log.append(g.detach())
            return h_new
        return _hook

    def _set_char_grad(self, requires_grad: bool):
        """Same method name as DarijaCharCNN so train_model's staging works."""
        for p in self.char_cnn.parameters():
            p.requires_grad = requires_grad
        for p in self.adapters.parameters():
            p.requires_grad = requires_grad

    def forward(self, input_ids, attention_mask, char_ids, **kw):
        if self._track_gates:
            self._gate_log = []
        self._char_feat = self.char_cnn(char_ids) if self._char_active else None
        out = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
        self._char_feat = None
        return self.classifier(out.last_hidden_state[:, 0])


# Parameter count check
_tmp = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
n_backbone = sum(p.numel() for p in _tmp.xlmr.parameters())
n_char     = sum(p.numel() for p in _tmp.char_cnn.parameters())
n_adapters = sum(p.numel() for p in _tmp.adapters.parameters())
n_total    = sum(p.numel() for p in _tmp.parameters())
print(f"M2 parameter breakdown:")
print(f"  XLM-R backbone : {n_backbone:>12,}")
print(f"  CharCNN        : {n_char:>12,}")
print(f"  12 adapters    : {n_adapters:>12,}  ← NEW")
print(f"  Total          : {n_total:>12,}")
print(f"  Overhead       : {100*(n_total - n_backbone)/n_backbone:.2f}%")
del _tmp
import gc; gc.collect()
print("✅ M2 model class defined")

In [ ]:
# ── Patch make_optimizer to handle M2's adapters group ────────────────
# Backward compatible: still works for M0 (no extensions) and M1 (char_proj).

def make_optimizer(model: nn.Module) -> AdamW:
    named = list(model.named_parameters())
    xlmr_p      = [p for n, p in named
                   if n.startswith("xlmr.") and "char_proj" not in n]
    char_proj_p = [p for n, p in named if "char_proj" in n]
    adapter_p   = [p for n, p in named if n.startswith("adapters.")]
    head_p      = [p for n, p in named
                   if not n.startswith("xlmr.")
                   and "char_proj" not in n
                   and not n.startswith("adapters.")]
    groups = [{"params": xlmr_p, "lr": 2e-5}]
    if char_proj_p: groups.append({"params": char_proj_p, "lr": 4e-5})
    if adapter_p:   groups.append({"params": adapter_p,   "lr": 5e-5})
    if head_p:      groups.append({"params": head_p,      "lr": 5e-5})
    print(f"  Optimizer groups: xlmr={len(xlmr_p)}, "
          f"char_proj={len(char_proj_p)}, "
          f"adapter={len(adapter_p)}, head={len(head_p)}")
    return AdamW(groups, weight_decay=Config.WEIGHT_DECAY)

print("✅ make_optimizer patched (M2-aware, backward compatible with M0/M1)")

In [ ]:
# ── Training engine (compute_metrics, train_model, etc.) ──────────────

def compute_metrics(labels, preds, name="") -> dict:
    labels, preds = np.array(labels), np.array(preds)
    macro_f1  = f1_score(labels, preds, average="macro", zero_division=0)
    per_class = f1_score(labels, preds, average=None, zero_division=0,
                         labels=[0,1,2])
    qwk = cohen_kappa_score(labels, preds, weights="quadratic")
    mae = float(np.mean(np.abs(labels - preds)))
    if name:
        W = 54
        print(f"\n{'═'*W}\n  {name}\n{'═'*W}")
        print(f"  Macro-F1 : {macro_f1:.4f}")
        print(f"  QWK      : {qwk:.4f}")
        print(f"  MAE      : {mae:.4f}")
        print(f"  mild  F1 : {per_class[0]:.4f}")
        print(f"  mod   F1 : {per_class[1]:.4f}")
        print(f"  sev   F1 : {per_class[2]:.4f}")
    return dict(macro_f1=macro_f1, qwk=qwk, mae=mae,
                mild_f1=per_class[0], moderate_f1=per_class[1],
                severe_f1=per_class[2])


def _forward(model, batch):
    ids   = batch["input_ids"].to(Config.DEVICE)
    mask  = batch["attention_mask"].to(Config.DEVICE)
    chars = batch["char_ids"].to(Config.DEVICE)
    lbls  = batch["label"].to(Config.DEVICE)
    return model(input_ids=ids, attention_mask=mask, char_ids=chars), lbls


def _train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        loss = corn_loss(logits, lbls, num_classes=Config.NUM_CLASSES)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), Config.CLIP_GRAD)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return total_loss/len(loader), f1_score(lbls_all, preds_all,
                                            average="macro", zero_division=0)


@torch.no_grad()
def _eval_epoch(model, loader):
    model.eval()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        total_loss += corn_loss(logits, lbls,
                                num_classes=Config.NUM_CLASSES).item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return (total_loss/len(loader),
            f1_score(lbls_all, preds_all, average="macro", zero_division=0),
            preds_all, lbls_all)


def train_model(model, name: str, activate_at_epoch: int = 2):
    model = model.to(Config.DEVICE)
    optimizer  = make_optimizer(model)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, total_steps//10, total_steps)

    best_f1, best_preds, best_labels = 0.0, None, None
    no_improve = 0
    ckpt = os.path.join(Config.SAVE_DIR, f"{name.replace(' ','_')}.pt")

    W = 70
    print(f"\n{'═'*W}\n  {name}")
    print(f"  Steps/epoch : {len(train_loader)}   "
          f"Total : {total_steps}   Warmup : {total_steps//10}")
    print(f"{'═'*W}")
    print(f"  {'Ep':>3}  {'TrLoss':>8}  {'TrF1':>7}  "
          f"{'VlLoss':>8}  {'VlF1':>7}  {'Best':>4}")
    print(f"  {'─'*56}")

    for epoch in range(1, Config.EPOCHS + 1):
        if epoch == activate_at_epoch and hasattr(model, '_char_active'):
            if not model._char_active:
                model._char_active = True
                if hasattr(model, '_set_char_grad'):
                    model._set_char_grad(True)
                if hasattr(model, '_set_extension_grad'):
                    model._set_extension_grad(True)
                print(f"  ★ Extension modules ACTIVATED at epoch {epoch}")

        tr_loss, tr_f1 = _train_epoch(model, train_loader,
                                       optimizer, scheduler)
        vl_loss, vl_f1, preds, labels = _eval_epoch(model, val_loader)

        flag = ""
        if vl_f1 > best_f1:
            best_f1, best_preds, best_labels = vl_f1, preds, labels
            torch.save(model.state_dict(), ckpt)
            no_improve = 0; flag = " ✓"
        else:
            no_improve += 1

        print(f"  {epoch:>3}  {tr_loss:>8.4f}  {tr_f1:>7.4f}  "
              f"{vl_loss:>8.4f}  {vl_f1:>7.4f}{flag}")
        if no_improve >= Config.PATIENCE:
            print(f"\n  Early stop — best val F1 = {best_f1:.4f}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
    print(f"  Best checkpoint restored → val Macro-F1 = {best_f1:.4f}")
    return model, best_f1, best_preds, best_labels


print("✅ Training engine ready")

In [ ]:
import gc, json, os
m2_val_records  = []
m2_test_records = []
m2_checkpoints  = {}
print("✅ M2 result containers initialized")

In [ ]:
# ── M2 · seed 42 ──────────────────────────────────────────────────────
SEED = 42

try: del model
except: pass
gc.collect(); torch.cuda.empty_cache()

seed_everything(SEED)
model = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
model, _, val_preds, val_labels = train_model(
    model, f"M2_CharGatedAdapter_seed{SEED}")

vm = compute_metrics(val_labels, val_preds, name=f"M2 seed{SEED} VAL")
m2_val_records.append(vm)

_, _, te_preds, te_labels = _eval_epoch(model, test_loader)
tm = compute_metrics(te_labels, te_preds, name=f"M2 seed{SEED} TEST")
m2_test_records.append(tm)

m2_checkpoints[SEED] = f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed{SEED}.pt"

# Save partial results immediately (in case next seed crashes)
with open(f"{Config.SAVE_DIR}/m2_partial.json", "w") as f:
    json.dump({"val": m2_val_records, "test": m2_test_records,
               "seeds_done": [SEED]}, f, indent=2)

del model
gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Seed {SEED} done · partial results saved")

In [ ]:
# ── M2 · seed 0 ───────────────────────────────────────────────────────
SEED = 0

try: del model
except: pass
gc.collect(); torch.cuda.empty_cache()

seed_everything(SEED)
model = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
model, _, val_preds, val_labels = train_model(
    model, f"M2_CharGatedAdapter_seed{SEED}")

vm = compute_metrics(val_labels, val_preds, name=f"M2 seed{SEED} VAL")
m2_val_records.append(vm)

_, _, te_preds, te_labels = _eval_epoch(model, test_loader)
tm = compute_metrics(te_labels, te_preds, name=f"M2 seed{SEED} TEST")
m2_test_records.append(tm)

m2_checkpoints[SEED] = f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed{SEED}.pt"

with open(f"{Config.SAVE_DIR}/m2_partial.json", "w") as f:
    json.dump({"val": m2_val_records, "test": m2_test_records,
               "seeds_done": [42, SEED]}, f, indent=2)

del model
gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Seed {SEED} done · partial results saved")

In [ ]:
# ── Hard memory reset before seed 1 ───────────────────────────────────
import gc, torch

# Clear any lingering model references
for name in list(globals()):
    obj = globals().get(name)
    if isinstance(obj, torch.nn.Module):
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

# Report state
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")
print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

In [ ]:
SEED = 1

try: del model
except: pass
gc.collect(); torch.cuda.empty_cache()

seed_everything(SEED)
model = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
model, _, val_preds, val_labels = train_model(
    model, f"M2_CharGatedAdapter_seed{SEED}")

vm = compute_metrics(val_labels, val_preds, name=f"M2 seed{SEED} VAL")
m2_val_records.append(vm)

_, _, te_preds, te_labels = _eval_epoch(model, test_loader)
tm = compute_metrics(te_labels, te_preds, name=f"M2 seed{SEED} TEST")
m2_test_records.append(tm)

m2_checkpoints[SEED] = f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed{SEED}.pt"

with open(f"{Config.SAVE_DIR}/m2_partial.json", "w") as f:
    json.dump({"val": m2_val_records, "test": m2_test_records,
               "seeds_done": [42, SEED]}, f, indent=2)

del model
gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ Seed {SEED} done · partial results saved")    

In [ ]:
# ── Gate value analysis (M2 seed 42 — strongest interpretability evidence) ──
import numpy as np

# Reload seed 42 with track_gates=True
seed_everything(42)
m2_for_gates = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab)).to(Config.DEVICE)
m2_for_gates.load_state_dict(torch.load(
    f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed42.pt",
    map_location=Config.DEVICE))
m2_for_gates._char_active = True


@torch.no_grad()
def collect_gates(model, loader, char_vocab):
    model.eval()
    model._track_gates = True
    arabizi_ids = {char_vocab[d] for d in "23579" if d in char_vocab}
    n_layers = len(model.adapters)
    
    ar_sum = torch.zeros(n_layers);  ar_cnt = torch.zeros(n_layers)
    pl_sum = torch.zeros(n_layers);  pl_cnt = torch.zeros(n_layers)
    
    for batch in loader:
        ids   = batch["input_ids"].to(Config.DEVICE)
        mask  = batch["attention_mask"].to(Config.DEVICE)
        chars = batch["char_ids"].to(Config.DEVICE)
        _ = model(input_ids=ids, attention_mask=mask, char_ids=chars)
        gates = torch.stack(model._gate_log, dim=0)  # [L, B, T]
        is_ar = torch.zeros_like(chars[..., 0], dtype=torch.bool)
        for did in arabizi_ids:
            is_ar |= (chars == did).any(dim=-1)
        real = mask.bool()
        is_ar = (is_ar & real).cpu()
        is_pl = (real & ~is_ar.to(real.device)).cpu()
        for L in range(n_layers):
            g = gates[L].cpu()
            ar_sum[L] += g[is_ar].sum();  ar_cnt[L] += is_ar.sum()
            pl_sum[L] += g[is_pl].sum();  pl_cnt[L] += is_pl.sum()
    
    model._track_gates = False; model._gate_log = []
    return (ar_sum / ar_cnt.clamp(min=1)).numpy(), (pl_sum / pl_cnt.clamp(min=1)).numpy()


# Run on val set (more tokens than test = more statistical power)
g_ar, g_pl = collect_gates(m2_for_gates, val_loader, char_vocab)

print("=" * 64)
print("  GATE VALUE ANALYSIS — M2 seed 42 (validation set)")
print("  Gate > 0.5 = adapter applied more strongly")
print("=" * 64)
print(f"\n  {'Layer':>5} | {'Arabizi':>10} | {'Plain':>10} | {'Δ':>8} | {'  '}")
print("  " + "─" * 52)
for L in range(len(g_ar)):
    d = g_ar[L] - g_pl[L]
    flag = " ★" if d > 0.01 else "  "
    print(f"  {L+1:>5} | {g_ar[L]:>10.4f} | {g_pl[L]:>10.4f} | {d:>+7.4f}{flag}")

mean_diff = (g_ar - g_pl).mean()
n_positive = (g_ar > g_pl).sum()
print(f"\n  Layers where Arabizi gate > Plain gate : {n_positive}/12")
print(f"  Mean gate difference (Arabizi − Plain) : {mean_diff:+.4f}")
print(f"  Max gate difference at layer           : {(g_ar - g_pl).argmax() + 1}")

# Save for paper figure
np.save(f"{Config.SAVE_DIR}/gate_arabizi.npy", g_ar)
np.save(f"{Config.SAVE_DIR}/gate_plain.npy", g_pl)
print(f"\n  ✓ Saved gate_arabizi.npy and gate_plain.npy")
print("=" * 64)

del m2_for_gates
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── Gate value analysis (M2 seed 1 — the strongest seed) ─────────────
import numpy as np, gc

try: del m2_for_gates
except: pass
gc.collect(); torch.cuda.empty_cache()

seed_everything(1)
m2_for_gates = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab)).to(Config.DEVICE)
m2_for_gates.load_state_dict(torch.load(
    f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed1.pt",
    map_location=Config.DEVICE))
m2_for_gates._char_active = True


@torch.no_grad()
def collect_gates(model, loader, char_vocab):
    model.eval()
    model._track_gates = True
    arabizi_ids = {char_vocab[d] for d in "23579" if d in char_vocab}
    n_layers = len(model.adapters)
    ar_sum = torch.zeros(n_layers); ar_cnt = torch.zeros(n_layers)
    pl_sum = torch.zeros(n_layers); pl_cnt = torch.zeros(n_layers)
    
    for batch in loader:
        ids   = batch["input_ids"].to(Config.DEVICE)
        mask  = batch["attention_mask"].to(Config.DEVICE)
        chars = batch["char_ids"].to(Config.DEVICE)
        _ = model(input_ids=ids, attention_mask=mask, char_ids=chars)
        gates = torch.stack(model._gate_log, dim=0)  # [L, B, T]
        is_ar = torch.zeros_like(chars[..., 0], dtype=torch.bool)
        for did in arabizi_ids:
            is_ar |= (chars == did).any(dim=-1)
        real = mask.bool()
        is_ar_cpu = (is_ar & real).cpu()
        is_pl_cpu = (real.cpu() & ~is_ar_cpu)
        for L in range(n_layers):
            g = gates[L].cpu()
            ar_sum[L] += g[is_ar_cpu].sum(); ar_cnt[L] += is_ar_cpu.sum()
            pl_sum[L] += g[is_pl_cpu].sum(); pl_cnt[L] += is_pl_cpu.sum()
    
    model._track_gates = False; model._gate_log = []
    return (ar_sum / ar_cnt.clamp(min=1)).numpy(), (pl_sum / pl_cnt.clamp(min=1)).numpy()


g_ar, g_pl = collect_gates(m2_for_gates, val_loader, char_vocab)

print("=" * 64)
print("  GATE VALUE ANALYSIS — M2 seed 1 (validation set)")
print("  Gate > 0.5 = adapter applied more strongly")
print("=" * 64)
print(f"\n  {'Layer':>5} | {'Arabizi':>10} | {'Plain':>10} | {'Δ':>8}")
print("  " + "─" * 50)
for L in range(len(g_ar)):
    d = g_ar[L] - g_pl[L]
    flag = " ★" if d > 0.01 else "  "
    print(f"  {L+1:>5} | {g_ar[L]:>10.4f} | {g_pl[L]:>10.4f} | {d:>+7.4f}{flag}")

mean_diff = (g_ar - g_pl).mean()
n_positive = int((g_ar > g_pl).sum())
print(f"\n  Layers where Arabizi gate > Plain gate : {n_positive}/12")
print(f"  Mean gate difference (Arabizi − Plain) : {mean_diff:+.4f}")
print(f"  Max difference at layer                : {int((g_ar - g_pl).argmax()) + 1}")

np.save(f"{Config.SAVE_DIR}/gate_arabizi.npy", g_ar)
np.save(f"{Config.SAVE_DIR}/gate_plain.npy", g_pl)
print(f"\n  ✓ Saved gate_arabizi.npy and gate_plain.npy")
print("=" * 64)

In [ ]:
# ── Save gate analysis figure ────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

g_ar = np.load(f"{Config.SAVE_DIR}/gate_arabizi.npy")
g_pl = np.load(f"{Config.SAVE_DIR}/gate_plain.npy")

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(1, 13)
width = 0.36

bars1 = ax.bar(x - width/2, g_ar, width, label="Arabizi tokens",
               color="#d97706", edgecolor="black", linewidth=0.6)
bars2 = ax.bar(x + width/2, g_pl, width, label="Plain tokens",
               color="#475569", edgecolor="black", linewidth=0.6)

ax.axhline(0.5, ls="--", c="gray", alpha=0.6, lw=1, label="Neutral (0.5)")
ax.set_xticks(x)
ax.set_xticklabels([f"L{i}" for i in x])
ax.set_xlabel("XLM-RoBERTa encoder layer", fontsize=11)
ax.set_ylabel("Mean gate activation", fontsize=11)
ax.set_title("M2 — Character-Gated Adapter activations per encoder layer\n"
             "(11 of 12 layers route Arabizi tokens through adapters more strongly)",
             fontsize=11)
ax.legend(loc="lower left", framealpha=0.95)
ax.set_ylim(0, 0.7)
ax.grid(axis="y", alpha=0.3, linestyle=":")

plt.tight_layout()
plt.savefig(f"{Config.SAVE_DIR}/gate_values_M2_seed1.png", dpi=200, bbox_inches="tight")
plt.savefig(f"{Config.SAVE_DIR}/gate_values_M2_seed1.pdf", bbox_inches="tight")
plt.show()
print("✓ Saved gate_values_M2_seed1.png and .pdf")

In [ ]:
import json, os
with open(f"{Config.SAVE_DIR}/m2_partial.json") as f:
    partial = json.load(f)

m2_val_records  = partial["val"]
m2_test_records = partial["test"]
m2_checkpoints  = {
    42: f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed42.pt",
    0:  f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed0.pt",
}

print(f"✅ Recovered {len(m2_val_records)} completed seeds: 42, 0")
for i, vm in enumerate(m2_val_records):
    print(f"   Seed {[42,0][i]} val macro-F1: {vm['macro_f1']:.4f}")

In [ ]:
# ── Aggregate M2 results across 3 seeds ───────────────────────────────
def _agg(records):
    keys = records[0].keys()
    return {k: {"mean": float(np.mean([r[k] for r in records])),
                "std":  float(np.std( [r[k] for r in records])),
                "runs": [r[k] for r in records]} for k in keys}

m2_results = {"val": _agg(m2_val_records),
              "test": _agg(m2_test_records),
              "checkpoints": m2_checkpoints}

with open(f"{Config.SAVE_DIR}/m2_results.json", "w") as f:
    json.dump({k: m2_results[k] for k in ["val","test"]}, f, indent=2)

print("="*70)
print("  M2 — FINAL AGGREGATE (3 seeds)")
print("="*70)
for split in ["val", "test"]:
    r = m2_results[split]
    print(f"\n  {split.upper()}:")
    print(f"    Macro-F1   : {r['macro_f1']['mean']:.4f} ± {r['macro_f1']['std']:.4f}")
    print(f"    QWK        : {r['qwk']['mean']:.4f} ± {r['qwk']['std']:.4f}")
    print(f"    MAE        : {r['mae']['mean']:.4f} ± {r['mae']['std']:.4f}")
    print(f"    Mild F1    : {r['mild_f1']['mean']:.4f}")
    print(f"    Mod  F1    : {r['moderate_f1']['mean']:.4f}")
    print(f"    Sev  F1    : {r['severe_f1']['mean']:.4f} ± {r['severe_f1']['std']:.4f}")
    print(f"    Per seed   : {[f'{x:.4f}' for x in r['macro_f1']['runs']]}")
print("\n✅ m2_results.json saved")

In [ ]:
# ── M2 — 3 seeds ──────────────────────────────────────────────────────

import gc, json
SEEDS = [42, 0, 1]

print("█"*70)
print("  M2 — Character-Gated Adapter · 3 seeds")
print("█"*70)

m2_val_records, m2_test_records, m2_checkpoints = [], [], {}

for seed in SEEDS:
    seed_everything(seed)
    model = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
    model, _, val_preds, val_labels = train_model(
        model, f"M2_CharGatedAdapter_seed{seed}")
    
    vm = compute_metrics(val_labels, val_preds,
                          name=f"M2 seed{seed} VAL")
    m2_val_records.append(vm)
    
    _, _, te_preds, te_labels = _eval_epoch(model, test_loader)
    tm = compute_metrics(te_labels, te_preds,
                          name=f"M2 seed{seed} TEST")
    m2_test_records.append(tm)
    
    m2_checkpoints[seed] = f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed{seed}.pt"
    
    del model
    gc.collect()
    torch.cuda.empty_cache()


def _agg(records):
    keys = records[0].keys()
    return {k: {"mean": float(np.mean([r[k] for r in records])),
                "std":  float(np.std( [r[k] for r in records])),
                "runs": [r[k] for r in records]} for k in keys}


m2_results = {"val": _agg(m2_val_records),
              "test": _agg(m2_test_records),
              "checkpoints": m2_checkpoints}

with open(f"{Config.SAVE_DIR}/m2_results.json", "w") as f:
    json.dump({k: m2_results[k] for k in ["val", "test"]}, f, indent=2)

print("\n" + "="*70)
print("  ✅ M2 COMPLETE")
print("="*70)
print(f"  Val Macro-F1  : {m2_results['val']['macro_f1']['mean']:.4f} ± "
      f"{m2_results['val']['macro_f1']['std']:.4f}")
print(f"  Val per seed  : {[f'{x:.4f}' for x in m2_results['val']['macro_f1']['runs']]}")
print(f"  Val Severe F1 : {m2_results['val']['severe_f1']['mean']:.4f} ± "
      f"{m2_results['val']['severe_f1']['std']:.4f}")
print(f"  Val QWK       : {m2_results['val']['qwk']['mean']:.4f} ± "
      f"{m2_results['val']['qwk']['std']:.4f}")
print(f"\n  Test Macro-F1 : {m2_results['test']['macro_f1']['mean']:.4f} ± "
      f"{m2_results['test']['macro_f1']['std']:.4f}")
print(f"  Saved to      : {Config.SAVE_DIR}/m2_results.json")
print("="*70)

In [ ]:
print("char_vocab :", type(char_vocab).__name__, len(char_vocab))
print("train_loader:", len(train_loader), "batches")
print("val_loader  :", len(val_loader),   "batches")
print("test_loader :", len(test_loader),  "batches")
print("train_model :", callable(train_model))
print("compute_metrics :", callable(compute_metrics))
print("CharCNN :", CharCNN)
print("CoralLayer :", CoralLayer)

# data relabeling test

In [ ]:
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)
print("char_vocab:", len(char_vocab))
print("train_loader:", len(train_loader), "batches")

In [ ]:
# Quick look at what columns and values we have
print("TRAIN columns:", train_df.columns.tolist())
print("VAL columns  :", val_df.columns.tolist())
print()
print("TRAIN label dist  :", train_df['label'].value_counts().sort_index().to_dict())
if 'weak_label' in train_df.columns:
    print("TRAIN weak_label  :", train_df['weak_label'].value_counts().to_dict())
print()
print("TRAIN sample post:")
print(repr(train_df.iloc[0]['text'][:200]))
print(f"label={train_df.iloc[0]['label']}, "
      f"weak_label={train_df.iloc[0].get('weak_label', 'n/a')}")

In [ ]:
# ── Improved weak supervision: lexicon + negation + sarcasm filters ──
import re, pandas as pd, numpy as np
from collections import Counter

# ── Improved severity lexicon
IMPROVED_LEXICON = {
    'mild': [
        # English
        'stressed', 'stress', 'nervous', 'worried', 'anxious', 'anxiety',
        'overthinking', 'struggling', 'burnout', 'not okay', 'feeling down',
        'overwhelmed', 'restless', 'on edge', 'cant focus', "can't focus",
        # French
        'nerveux', 'nerveuse', 'perdu', 'perdue', 'fatigue', 'fatigué',
        'fatiguée', 'angoissé', 'angoissée', 'inquiet', 'inquiète',
        # Darija / Arabizi
        'qalaq', 'm9lq', 'm9l9', 'khayef', 'khayfa', '3yit', 'mfkr',
        'tbnt', 'mrhq', 'ta3ban', 'ta3bana',
        # Arabic
        'قلق', 'متوتر', 'خايف', 'حزين',
    ],
    'moderate': [
        # English
        'lonely', 'sad', 'hopeless', 'depressed', 'depression', 'worthless',
        'empty', 'numb', 'breakdown', 'give up', 'gave up', 'no motivation',
        'isolated', 'withdrawn', 'cant sleep', "can't sleep", 'insomnia',
        'crying', 'cried', 'tears',
        # French
        'seul', 'seule', 'triste', 'vide', 'désespéré', 'désespérée',
        'déprimé', 'déprimée', 'sans espoir', 'sans motivation',
        # Darija / Arabizi
        'wahid', 'wahdani', 'wahda', 'bohdi', '7azin', '7azina', 'mafi aml',
        'bkait', 'kanbki', 'mkalakh', 'mkhnoq', 'kan7ess b faragh',
        'rasi khawi', 'ma bqitch n7ess', "ma3ndich motivation",
        'ma3ndich rghba', 'f9edt l amal',
        # Arabic
        'بوحدي', 'وحدي', 'كنبكي', 'عييت', 'تعبان', 'ماعنديش أمل',
        'ماعنديش رغبة', 'فقدت الأمل',
    ],
    'severe': [
        # English
        'suicide', 'suicidal', 'kill myself', 'killing myself',
        'end my life', 'end it all', 'no reason to live',
        'want to die', 'wanna die', 'self harm', 'self-harm',
        'cutting myself', 'overdose', 'jump off',
        # Darija / Arabizi
        'bghit nmout', 'bghit nmot', 'bghit nmt', 'khir nmout',
        'mabghitch n3ich', 'ma bghitch n3ich', 'krht 7yati',
        'krht hyati', 'ma 3ndi 7ayat', 'ma3ndich 7ayat', 'intihar',
        # Arabic
        'انتحار', 'بغيت نموت', 'كرهت حياتي', 'ما بغيتش نعيش',
        'أريد الموت',
    ],
}

# Negation markers — within 4 words BEFORE the keyword
NEGATION_MARKERS = {
    "not", "no", "don't", "dont", "doesn't", "doesnt", "didn't", "didnt",
    "never", "nobody", "nothing", "isn't", "isnt", "wasn't", "wasnt",
    "won't", "wont", "wouldn't", "wouldnt", "ne", "pas", "jamais",
    "rien", "personne", "aucun", "aucune", "ma", "mch", "mach", "machi",
    "mabghitch", "mabqitch", "ma3ndich", "ma3reftch",
    "لا", "ليس", "ليست", "لم", "لن", "ولا",
}

# Sarcasm / metaphor context anywhere in post
SARCASM_MARKERS = [
    "would kill for", "would die for", "would die if", "killing it",
    "to die for", "dying to", "dying for", "feels like dying",
    "lol", "lmao", "haha", "😂", "🤣",
    "joking", "kidding", "sarcasm", "sarcastic",
    "song", "lyrics", "movie", "film", "book quote",
    "metaphor", "metaphorically",
]

# Reported speech within 50 chars before keyword
REPORTED_MARKERS = [
    "she said", "he said", "they said", "told me", "told him", "told her",
    "wrote that", "posted that", "tweeted",
    "il a dit", "elle a dit", "qal", "qalat", "قال", "قالت",
]


def _has_negation_before(text_lower: str, kw_lower: str, window: int = 4) -> bool:
    """Check 4-word window BEFORE every occurrence of kw."""
    idx = 0
    while True:
        idx = text_lower.find(kw_lower, idx)
        if idx == -1:
            return False
        before_text = text_lower[:idx]
        before_words = before_text.split()
        window_words = before_words[-window:] if before_words else []
        if any(neg in window_words for neg in NEGATION_MARKERS):
            return True
        idx += len(kw_lower)


def _has_sarcasm(text_lower: str) -> bool:
    return any(m in text_lower for m in SARCASM_MARKERS)


def _has_reported_speech(text_lower: str, kw_lower: str) -> bool:
    idx = text_lower.find(kw_lower)
    if idx == -1: return False
    context = text_lower[max(0, idx-50):idx]
    return any(m in context for m in REPORTED_MARKERS)


def improved_label(text: str):
    """Return (label_int, reasons_list). label=-1 means ambiguous → drop."""
    if not isinstance(text, str) or len(text) < 10:
        return -1, ["too_short"]
    
    t = text.lower()
    reasons = []
    severe_hits, moderate_hits, mild_hits = [], [], []
    sarcastic = _has_sarcasm(t)
    
    for severity in ['severe', 'moderate', 'mild']:
        for kw in IMPROVED_LEXICON[severity]:
            kw_l = kw.lower()
            if kw_l not in t:
                continue
            if _has_negation_before(t, kw_l):
                reasons.append(f"neg:{kw}")
                continue
            if severity == 'severe' and sarcastic:
                reasons.append(f"sarc:{kw}")
                continue
            if severity == 'severe' and _has_reported_speech(t, kw_l):
                reasons.append(f"rep:{kw}")
                continue
            if   severity == 'severe':   severe_hits.append(kw)
            elif severity == 'moderate': moderate_hits.append(kw)
            else:                        mild_hits.append(kw)
    
    n_sev = len(severe_hits)
    n_mod = len(moderate_hits)
    n_mld = len(mild_hits)
    
    if n_sev >= 2:
        return 2, [f"sev:{n_sev}"] + reasons
    if n_sev == 1 and n_mod >= 1:
        return 2, ["sev:1+mod"] + reasons
    if n_sev == 1:
        return 1, ["sev:1_downgraded"] + reasons
    if n_mod >= 1:
        return 1, [f"mod:{n_mod}"] + reasons
    if n_mld >= 1:
        return 0, [f"mild:{n_mld}"] + reasons
    return -1, ["no_hits"] + reasons


# ── Apply to train and val (NOT test — test is gold human labels)
print("=" * 64)
print("  APPLYING IMPROVED LEXICON")
print("=" * 64)

for split_name, df in [("train", train_df), ("val", val_df)]:
    old = df['label'].copy()
    results = df['text'].apply(improved_label)
    df['label_new'] = results.apply(lambda x: x[0])
    df['reasons']   = results.apply(lambda x: ";".join(x[1][:3]))
    
    changed = (df['label_new'] != old).sum()
    ambiguous = (df['label_new'] == -1).sum()
    
    print(f"\n  [{split_name}] {len(df):,} posts")
    print(f"    Labels changed   : {changed:,} ({100*changed/len(df):.1f}%)")
    print(f"    Ambiguous (drop) : {ambiguous:,} ({100*ambiguous/len(df):.1f}%)")
    
    cm = pd.crosstab(old, df['label_new'].clip(lower=-1),
                     rownames=['old'], colnames=['new'], dropna=False)
    print(f"\n    Crosstab (old → new):")
    print(cm.to_string())
    
    new_dist = df['label_new'].value_counts().sort_index().to_dict()
    print(f"\n    New label distribution: {new_dist}")
    print(f"    Old label distribution: {old.value_counts().sort_index().to_dict()}")

print("\n" + "=" * 64)
print("  ⚠ NOTHING SAVED YET. Inspect the crosstab above.")
print("  Then run the next cell to spot-check 15 random changed posts.")
print("=" * 64)

In [ ]:
# ── Conservative lexicon refinement (Option 2) ────────────────────────
# Strategy: only DOWNGRADE/REMOVE clearly wrong severe labels.
#   - Negation before severe keyword → downgrade severe to moderate
#   - Sarcasm context with severe keyword → downgrade severe to moderate
#   - Reported speech with severe keyword → downgrade severe to moderate
# Everything else keeps the original label. No ambiguous drops.

import re, pandas as pd

SEVERE_KEYWORDS = [
    # English
    'suicide', 'suicidal', 'kill myself', 'killing myself',
    'end my life', 'end it all', 'no reason to live',
    'want to die', 'wanna die', 'self harm', 'self-harm',
    'cutting myself', 'overdose', 'jump off',
    # Darija / Arabizi
    'bghit nmout', 'bghit nmot', 'bghit nmt', 'khir nmout',
    'mabghitch n3ich', 'ma bghitch n3ich', 'krht 7yati',
    'krht hyati', 'ma 3ndi 7ayat', 'ma3ndich 7ayat', 'intihar',
    # Arabic
    'انتحار', 'بغيت نموت', 'كرهت حياتي', 'ما بغيتش نعيش',
    'أريد الموت',
]

NEGATION_MARKERS = {
    "not", "no", "don't", "dont", "doesn't", "doesnt", "didn't", "didnt",
    "never", "nobody", "nothing", "isn't", "isnt", "wasn't", "wasnt",
    "won't", "wont", "wouldn't", "wouldnt", "ne", "pas", "jamais",
    "rien", "personne", "aucun", "aucune", "ma", "mch", "mach", "machi",
    "mabghitch", "mabqitch", "ma3ndich",
    "لا", "ليس", "ليست", "لم", "لن", "ولا",
}

SARCASM_MARKERS = [
    "would kill for", "would die for", "would die if", "killing it",
    "to die for", "dying to", "dying for", "feels like dying",
    "lol", "lmao", "haha", "😂", "🤣",
    "joking", "kidding", "sarcasm", "sarcastic",
    "song lyric", "lyrics", "movie quote", "book quote",
    "metaphorically",
]

REPORTED_MARKERS = [
    "she said", "he said", "they said", "told me", "told him", "told her",
    "wrote that", "posted that", "tweeted",
    "il a dit", "elle a dit", "qal", "qalat", "قال", "قالت",
]


def _has_negation_before(t: str, kw: str, window: int = 4) -> bool:
    idx = 0
    while True:
        idx = t.find(kw, idx)
        if idx == -1:
            return False
        before_words = t[:idx].split()[-window:]
        if any(neg in before_words for neg in NEGATION_MARKERS):
            return True
        idx += len(kw)


def _has_sarcasm(t: str) -> bool:
    return any(m in t for m in SARCASM_MARKERS)


def _has_reported_speech(t: str, kw: str) -> bool:
    idx = t.find(kw)
    if idx == -1:
        return False
    return any(m in t[max(0, idx-50):idx] for m in REPORTED_MARKERS)


def refine_severe(text: str, current_label: int) -> tuple:
    """Conservative: only check whether SEVERE labels (2) are wrong.
    Returns (new_label, reason). Returns current_label unchanged for labels 0,1.
    """
    if current_label != 2:
        return current_label, "non-severe-kept"
    if not isinstance(text, str):
        return current_label, "no-text"
    
    t = text.lower()
    
    # Find which severe keyword(s) triggered the label
    matched = [kw for kw in SEVERE_KEYWORDS if kw.lower() in t]
    if not matched:
        # Severe-labeled but no severe keyword found in text — keep as is
        # (lexicon may have changed since labeling)
        return current_label, "no-severe-kw-found"
    
    # For each matched severe keyword, check filters
    all_negated = True
    all_sarcastic = True
    all_reported = True
    
    for kw in matched:
        kw_l = kw.lower()
        if not _has_negation_before(t, kw_l):
            all_negated = False
        if not _has_sarcasm(t):
            all_sarcastic = False
        if not _has_reported_speech(t, kw_l):
            all_reported = False
    
    if all_negated:
        return 1, "all-severe-kws-negated"
    if all_sarcastic:
        return 1, "sarcasm-context"
    if all_reported:
        return 1, "reported-speech"
    
    return current_label, "severe-kept"


# ── Apply conservative refinement
print("=" * 64)
print("  CONSERVATIVE REFINEMENT — only severe-labeled posts examined")
print("=" * 64)

# Clean up previous columns from earlier attempt
for df in [train_df, val_df]:
    for col in ['label_new', 'reasons']:
        if col in df.columns:
            df.drop(columns=[col], inplace=True)

for split_name, df in [("train", train_df), ("val", val_df)]:
    old = df['label'].copy()
    results = df.apply(lambda r: refine_severe(r['text'], r['label']), axis=1)
    df['label_new'] = results.apply(lambda x: x[0])
    df['reason']    = results.apply(lambda x: x[1])
    
    changed = (df['label_new'] != old).sum()
    severe_before = (old == 2).sum()
    severe_after  = (df['label_new'] == 2).sum()
    
    print(f"\n  [{split_name}] {len(df):,} posts")
    print(f"    Labels changed       : {changed:,} ({100*changed/len(df):.2f}%)")
    print(f"    Severe before        : {severe_before:,}")
    print(f"    Severe after         : {severe_after:,}")
    print(f"    Severe → Moderate    : {severe_before - severe_after:,}")
    
    if changed:
        # Reason breakdown for changed labels
        changed_reasons = df.loc[df['label_new'] != old, 'reason'].value_counts()
        print(f"\n    Why labels changed:")
        for reason, cnt in changed_reasons.items():
            print(f"      {reason:30s}: {cnt}")
    
    new_dist = df['label_new'].value_counts().sort_index().to_dict()
    old_dist = old.value_counts().sort_index().to_dict()
    print(f"\n    Old distribution: {old_dist}")
    print(f"    New distribution: {new_dist}")

print("\n" + "=" * 64)
print("  ⚠ NOTHING SAVED YET — inspect numbers above.")
print("  Expected: 1-5% label change, severe drops by 30-80 posts only.")
print("  If reasonable, run next cell to spot-check examples.")
print("=" * 64)

In [ ]:
# ── Spot-check: 10 posts whose label changed ─────────────────────────
import random
random.seed(42)

changed_train = train_df[train_df['label_new'] != train_df['label']].copy()
sample = changed_train.sample(min(10, len(changed_train)), random_state=42)

LABEL_NAMES = {0: 'mild', 1: 'moderate', 2: 'severe'}

print("=" * 70)
print("  SPOT-CHECK — 10 posts where label was changed")
print("=" * 70)
for i, (_, row) in enumerate(sample.iterrows(), 1):
    old_l = LABEL_NAMES[row['label']]
    new_l = LABEL_NAMES[row['label_new']]
    print(f"\n──[{i}/10]── {old_l} → {new_l}  (reason: {row['reason']})")
    text = row['text'][:400]
    print(f"  TEXT: {text}{'...' if len(row['text']) > 400 else ''}")
print("\n" + "=" * 70)

In [ ]:
print("train_df shape :", train_df.shape)
print("val_df shape   :", val_df.shape)
print("char_vocab size:", len(char_vocab))
print("train_loader   :", len(train_loader), "batches")
print()
print("CharCNN class  :", CharCNN if 'CharCNN' in dir() else "MISSING")
print("DarijaCharCNN  :", DarijaCharCNN if 'DarijaCharCNN' in dir() else "MISSING")
print("BaselineXLMR   :", BaselineXLMR if 'BaselineXLMR' in dir() else "MISSING")
print("train_model    :", "OK" if callable(train_model) else "MISSING")
print("compute_metrics:", "OK" if callable(compute_metrics) else "MISSING")
print("make_optimizer :", "OK" if callable(make_optimizer) else "MISSING")
print("_eval_epoch    :", "OK" if callable(_eval_epoch) else "MISSING")
print()
import torch
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

In [ ]:
class CharCNN(nn.Module):
    def __init__(self, vocab_size,
                 char_embed_dim=Config.CHAR_EMBED_DIM,
                 filter_sizes=Config.CHAR_FILTERS,
                 num_filters=Config.CHAR_NUM_FILTERS,
                 dropout=Config.CHAR_DROPOUT):
        super().__init__()
        self.char_embed = nn.Embedding(vocab_size, char_embed_dim,
                                        padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_embed_dim, num_filters, fs, padding=fs//2)
            for fs in filter_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.out_dim = num_filters * len(filter_sizes)

    def forward(self, char_ids):
        B, T, C = char_ids.shape
        x = self.char_embed(char_ids.view(B*T, C))
        x = x.transpose(1, 2)
        pooled = [torch.relu(conv(x)).max(dim=2).values for conv in self.convs]
        out = self.dropout(torch.cat(pooled, dim=-1))
        return out.view(B, T, -1)

print("✅ CharCNN defined")

In [ ]:
class BaselineXLMR(nn.Module):
    """M0 — Standard XLM-R + CORAL head."""
    def __init__(self, model_name=Config.MODEL_NAME,
                 num_classes=Config.NUM_CLASSES, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.xlmr(input_ids=input_ids,
                        attention_mask=attention_mask
                       ).last_hidden_state[:, 0]
        return self.classifier(cls)


class DarijaCharCNN(nn.Module):
    """M1 — XLM-R + CharCNN injected at embedding layer.
    
    Now accepts char_num_filters kwarg so we can run M1-wide for W2.
    """
    def __init__(self, model_name=Config.MODEL_NAME,
                 char_vocab_size=None,
                 num_classes=Config.NUM_CLASSES,
                 char_num_filters=Config.CHAR_NUM_FILTERS, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        _vs = char_vocab_size or len(char_vocab)
        # Allow override of num_filters for parameter-matched ablation (W2)
        self.char_cnn = CharCNN(_vs, num_filters=char_num_filters)
        self.char_proj = nn.Linear(self.char_cnn.out_dim, d)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            CoralLayer(d, num_classes),
        )
        nn.init.normal_(self.char_proj.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.char_proj.bias)

        self._char_active = False
        self._set_char_grad(False)
        self._char_feat = None

        def _emb_hook(module, input, output):
            if self._char_feat is not None and self._char_active:
                return output + self._char_feat
            return output
        self.xlmr.embeddings.word_embeddings.register_forward_hook(_emb_hook)

    def _set_char_grad(self, requires_grad):
        for p in self.char_cnn.parameters():  p.requires_grad = requires_grad
        for p in self.char_proj.parameters(): p.requires_grad = requires_grad

    def forward(self, input_ids, attention_mask, char_ids, **kwargs):
        self._char_feat = (self.char_proj(self.char_cnn(char_ids))
                            if self._char_active else None)
        outputs = self.xlmr(input_ids=input_ids,
                            attention_mask=attention_mask)
        self._char_feat = None
        return self.classifier(outputs.last_hidden_state[:, 0])


print("✅ BaselineXLMR and DarijaCharCNN defined")

In [ ]:
class BaselineXLMR(nn.Module):
    """M0 — Standard XLM-R + CORAL head."""
    def __init__(self, model_name=Config.MODEL_NAME,
                 num_classes=Config.NUM_CLASSES, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )

    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.xlmr(input_ids=input_ids,
                        attention_mask=attention_mask
                       ).last_hidden_state[:, 0]
        return self.classifier(cls)


class DarijaCharCNN(nn.Module):
    """M1 — XLM-R + CharCNN injected at embedding layer.
    
    Now accepts char_num_filters kwarg so we can run M1-wide for W2.
    """
    def __init__(self, model_name=Config.MODEL_NAME,
                 char_vocab_size=None,
                 num_classes=Config.NUM_CLASSES,
                 char_num_filters=Config.CHAR_NUM_FILTERS, **kwargs):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        _vs = char_vocab_size or len(char_vocab)
        # Allow override of num_filters for parameter-matched ablation (W2)
        self.char_cnn = CharCNN(_vs, num_filters=char_num_filters)
        self.char_proj = nn.Linear(self.char_cnn.out_dim, d)
        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            CoralLayer(d, num_classes),
        )
        nn.init.normal_(self.char_proj.weight, mean=0.0, std=0.01)
        nn.init.zeros_(self.char_proj.bias)

        self._char_active = False
        self._set_char_grad(False)
        self._char_feat = None

        def _emb_hook(module, input, output):
            if self._char_feat is not None and self._char_active:
                return output + self._char_feat
            return output
        self.xlmr.embeddings.word_embeddings.register_forward_hook(_emb_hook)

    def _set_char_grad(self, requires_grad):
        for p in self.char_cnn.parameters():  p.requires_grad = requires_grad
        for p in self.char_proj.parameters(): p.requires_grad = requires_grad

    def forward(self, input_ids, attention_mask, char_ids, **kwargs):
        self._char_feat = (self.char_proj(self.char_cnn(char_ids))
                            if self._char_active else None)
        outputs = self.xlmr(input_ids=input_ids,
                            attention_mask=attention_mask)
        self._char_feat = None
        return self.classifier(outputs.last_hidden_state[:, 0])


print("✅ BaselineXLMR and DarijaCharCNN defined")

In [ ]:
def make_optimizer(model: nn.Module) -> AdamW:
    named = list(model.named_parameters())
    xlmr_p      = [p for n, p in named
                   if n.startswith("xlmr.") and "char_proj" not in n]
    char_proj_p = [p for n, p in named if "char_proj" in n]
    char_cnn_p  = [p for n, p in named if n.startswith("char_cnn.")]
    head_p      = [p for n, p in named
                   if not n.startswith("xlmr.")
                   and "char_proj" not in n
                   and not n.startswith("char_cnn.")]
    groups = [{"params": xlmr_p, "lr": 2e-5}]
    if char_proj_p: groups.append({"params": char_proj_p, "lr": 4e-5})
    if char_cnn_p:  groups.append({"params": char_cnn_p,  "lr": 5e-5})
    if head_p:      groups.append({"params": head_p,      "lr": 5e-5})
    print(f"  Optimizer groups: xlmr={len(xlmr_p)}, "
          f"char_proj={len(char_proj_p)}, char_cnn={len(char_cnn_p)}, head={len(head_p)}")
    return AdamW(groups, weight_decay=Config.WEIGHT_DECAY)


def compute_metrics(labels, preds, name="") -> dict:
    labels, preds = np.array(labels), np.array(preds)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    per_class = f1_score(labels, preds, average=None, zero_division=0, labels=[0,1,2])
    qwk = cohen_kappa_score(labels, preds, weights="quadratic")
    mae = float(np.mean(np.abs(labels - preds)))
    if name:
        print(f"\n{'═'*54}\n  {name}\n{'═'*54}")
        print(f"  Macro-F1 : {macro_f1:.4f}")
        print(f"  QWK      : {qwk:.4f}")
        print(f"  MAE      : {mae:.4f}")
        print(f"  mild  F1 : {per_class[0]:.4f}")
        print(f"  mod   F1 : {per_class[1]:.4f}")
        print(f"  sev   F1 : {per_class[2]:.4f}")
    return dict(macro_f1=macro_f1, qwk=qwk, mae=mae,
                mild_f1=per_class[0], moderate_f1=per_class[1],
                severe_f1=per_class[2])


def _forward(model, batch):
    ids   = batch["input_ids"].to(Config.DEVICE)
    mask  = batch["attention_mask"].to(Config.DEVICE)
    chars = batch["char_ids"].to(Config.DEVICE)
    lbls  = batch["label"].to(Config.DEVICE)
    return model(input_ids=ids, attention_mask=mask, char_ids=chars), lbls


def _train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        loss = corn_loss(logits, lbls, num_classes=Config.NUM_CLASSES)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), Config.CLIP_GRAD)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return total_loss/len(loader), f1_score(lbls_all, preds_all,
                                            average="macro", zero_division=0)


@torch.no_grad()
def _eval_epoch(model, loader):
    model.eval()
    total_loss, preds_all, lbls_all = 0.0, [], []
    for batch in loader:
        logits, lbls = _forward(model, batch)
        total_loss += corn_loss(logits, lbls,
                                num_classes=Config.NUM_CLASSES).item()
        preds_all.extend(corn_label_from_logits(logits).cpu().tolist())
        lbls_all.extend(lbls.cpu().tolist())
    return (total_loss/len(loader),
            f1_score(lbls_all, preds_all, average="macro", zero_division=0),
            preds_all, lbls_all)


def train_model(model, name: str, activate_at_epoch: int = 2):
    model = model.to(Config.DEVICE)
    optimizer  = make_optimizer(model)
    total_steps = len(train_loader) * Config.EPOCHS
    scheduler  = get_linear_schedule_with_warmup(
        optimizer, total_steps//10, total_steps)

    best_f1 = 0.0; best_preds = best_labels = None
    no_improve = 0
    ckpt = os.path.join(Config.SAVE_DIR, f"{name.replace(' ','_')}.pt")

    print(f"\n{'═'*70}\n  {name}")
    print(f"  Steps/epoch : {len(train_loader)}   "
          f"Total : {total_steps}   Warmup : {total_steps//10}")
    print(f"{'═'*70}")
    print(f"  {'Ep':>3}  {'TrLoss':>8}  {'TrF1':>7}  "
          f"{'VlLoss':>8}  {'VlF1':>7}  {'Best':>4}")
    print(f"  {'─'*56}")

    for epoch in range(1, Config.EPOCHS + 1):
        if epoch == activate_at_epoch and hasattr(model, '_char_active'):
            if not model._char_active:
                model._char_active = True
                if hasattr(model, '_set_char_grad'):
                    model._set_char_grad(True)
                print(f"  ★ Extension modules ACTIVATED at epoch {epoch}")

        tr_loss, tr_f1 = _train_epoch(model, train_loader, optimizer, scheduler)
        vl_loss, vl_f1, preds, labels = _eval_epoch(model, val_loader)

        flag = ""
        if vl_f1 > best_f1:
            best_f1, best_preds, best_labels = vl_f1, preds, labels
            torch.save(model.state_dict(), ckpt)
            no_improve = 0; flag = " ✓"
        else:
            no_improve += 1

        print(f"  {epoch:>3}  {tr_loss:>8.4f}  {tr_f1:>7.4f}  "
              f"{vl_loss:>8.4f}  {vl_f1:>7.4f}{flag}")
        if no_improve >= Config.PATIENCE:
            print(f"\n  Early stop — best val F1 = {best_f1:.4f}")
            break

    model.load_state_dict(torch.load(ckpt, map_location=Config.DEVICE))
    print(f"  Best checkpoint restored → val Macro-F1 = {best_f1:.4f}")
    return model, best_f1, best_preds, best_labels


print("✅ Training engine ready")

In [ ]:
print("CharCNN class  :", "OK" if 'CharCNN' in dir() else "MISSING")
print("BaselineXLMR   :", "OK" if 'BaselineXLMR' in dir() else "MISSING")
print("DarijaCharCNN  :", "OK" if 'DarijaCharCNN' in dir() else "MISSING")
print("train_model    :", "OK" if callable(train_model) else "MISSING")
print("compute_metrics:", "OK" if callable(compute_metrics) else "MISSING")
print()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")
print(f"\nReady to launch W2 — M1-wide parameter-matched.")

In [ ]:
# ── W2: M1-wide — parameter-matched ablation (~1.2M extra params) ─────
import gc, json, time

# Clear any leftover models
try: del model
except: pass
gc.collect(); torch.cuda.empty_cache()

# M1-wide uses CharCNN with 256 filters per width (256 × 3 = 768-d output)
# instead of 64 (default).
# Param count: 369*32 + (2+3+4)*32*256 + (256*3)*768 + bias = ~671k
# Plus char_proj going from 192→768 to 768→768 adds: 768*768 = ~590k
# Total extra ≈ 1.26M, very close to M2's 1.2M adapter overhead.

print("=" * 70)
print("  W2 — M1-wide (parameter-matched to M2)")
print("=" * 70)

seed_everything(42)

# Build M1 with wider CharCNN
m1_wide = DarijaCharCNN(
    char_vocab_size=len(char_vocab),
    char_num_filters=256   # ← key change: 64 → 256
)

# Sanity check: parameter count
n_backbone = sum(p.numel() for p in m1_wide.xlmr.parameters())
n_char     = sum(p.numel() for p in m1_wide.char_cnn.parameters())
n_proj     = sum(p.numel() for p in m1_wide.char_proj.parameters())
n_head     = sum(p.numel() for p in m1_wide.classifier.parameters())
n_total    = sum(p.numel() for p in m1_wide.parameters())
print(f"\n  Parameter breakdown:")
print(f"    XLM-R backbone : {n_backbone:>12,}")
print(f"    CharCNN (wide) : {n_char:>12,}")
print(f"    char_proj      : {n_proj:>12,}")
print(f"    CORAL head     : {n_head:>12,}")
print(f"    ──────────────────────────────")
print(f"    Total          : {n_total:>12,}")
print(f"    Extra over M0  : {(n_total - n_backbone):>12,}  "
      f"(M2 had ~1.2M)")

# Train
t0 = time.time()
m1_wide, _, val_preds, val_labels = train_model(
    m1_wide, "W2_M1wide_seed42")
elapsed = (time.time() - t0) / 60
print(f"\n  Training time: {elapsed:.1f} min")

# Eval
vm = compute_metrics(val_labels, val_preds, name="W2 M1-wide VAL")

# Test eval
_, _, te_preds, te_labels = _eval_epoch(m1_wide, test_loader)
tm = compute_metrics(te_labels, te_preds, name="W2 M1-wide TEST")

# Save
w2_results = {
    "model": "M1-wide (char_num_filters=256)",
    "seed": 42,
    "param_count_total": n_total,
    "param_count_extra": n_total - n_backbone,
    "val": vm,
    "test": tm,
    "training_minutes": elapsed,
    "checkpoint": f"{Config.SAVE_DIR}/W2_M1wide_seed42.pt",
}
with open(f"{Config.SAVE_DIR}/w2_results.json", "w") as f:
    json.dump(w2_results, f, indent=2)

print("\n" + "=" * 70)
print("  ✅ W2 COMPLETE — saved to w2_results.json")
print("=" * 70)
print(f"  Comparison context:")
print(f"    M1 (paper, 64 filters)  : 0.7540 val F1, +0.18M params")
print(f"    M1-wide (256 filters)   : {vm['macro_f1']:.4f} val F1, "
      f"+{(n_total - n_backbone)/1e6:.2f}M params")
print(f"    M2 (12 adapters)        : 0.7434 val F1, +1.20M params")

del m1_wide
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── W1: AraBERT baseline (single seed, seed 42) ───────────────────────
# Why: closes the "why XLM-R, not an Arabic encoder?" question.
# Expected: AraBERT underperforms XLM-R on this corpus because most
# posts are heavily code-switched with French/English, which AraBERT
# wasn't trained on.

import gc, json, time
gc.collect(); torch.cuda.empty_cache()

ARABERT_NAME = "aubmindlab/bert-base-arabertv02"

print("=" * 70)
print(f"  W1 — AraBERT baseline ({ARABERT_NAME})")
print("=" * 70)

# Save XLM-R tokenizer reference so we can restore loaders later
_xlmr_tokenizer = tokenizer
_xlmr_train_loader = train_loader
_xlmr_val_loader = val_loader
_xlmr_test_loader = test_loader

# Build AraBERT tokenizer + retokenize data
print("\n  Loading AraBERT tokenizer...")
arabert_tokenizer = AutoTokenizer.from_pretrained(ARABERT_NAME)
print(f"  Vocab size: {arabert_tokenizer.vocab_size}")

print("  Re-tokenizing train/val/test for AraBERT...")
arabert_train_ds = DarijaDataset(train_df, arabert_tokenizer, char_vocab)
arabert_val_ds   = DarijaDataset(val_df,   arabert_tokenizer, char_vocab)
arabert_test_ds  = DarijaDataset(test_df,  arabert_tokenizer, char_vocab)

arabert_train_loader = DataLoader(arabert_train_ds, batch_size=Config.BATCH_SIZE,
                                   shuffle=True, num_workers=2, pin_memory=True)
arabert_val_loader   = DataLoader(arabert_val_ds, batch_size=Config.BATCH_SIZE*2,
                                   shuffle=False, num_workers=2, pin_memory=True)
arabert_test_loader  = DataLoader(arabert_test_ds, batch_size=Config.BATCH_SIZE*2,
                                   shuffle=False, num_workers=2, pin_memory=True)

# Swap loaders so train_model uses AraBERT-tokenized data
train_loader = arabert_train_loader
val_loader   = arabert_val_loader
test_loader  = arabert_test_loader

# AraBERT-flavored M0
class AraBERTBaseline(nn.Module):
    def __init__(self, model_name=ARABERT_NAME, num_classes=Config.NUM_CLASSES, **kw):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)  # name kept for compat
        d = self.xlmr.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            CoralLayer(d, num_classes),
        )
    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.xlmr(input_ids=input_ids,
                        attention_mask=attention_mask
                       ).last_hidden_state[:, 0]
        return self.classifier(cls)

# Train
seed_everything(42)
m_arabert = AraBERTBaseline()
print(f"\n  Backbone params : {sum(p.numel() for p in m_arabert.xlmr.parameters()):,}")
print(f"  Total params    : {sum(p.numel() for p in m_arabert.parameters()):,}")

t0 = time.time()
m_arabert, _, val_preds, val_labels = train_model(
    m_arabert, "W1_AraBERT_seed42")
elapsed = (time.time() - t0) / 60
print(f"\n  Training time: {elapsed:.1f} min")

vm = compute_metrics(val_labels, val_preds, name="W1 AraBERT VAL")
_, _, te_preds, te_labels = _eval_epoch(m_arabert, test_loader)
tm = compute_metrics(te_labels, te_preds, name="W1 AraBERT TEST")

w1_results = {
    "model": ARABERT_NAME,
    "seed": 42,
    "val": vm,
    "test": tm,
    "training_minutes": elapsed,
    "checkpoint": f"{Config.SAVE_DIR}/W1_AraBERT_seed42.pt",
}
with open(f"{Config.SAVE_DIR}/w1_results.json", "w") as f:
    json.dump(w1_results, f, indent=2)

print("\n" + "=" * 70)
print("  ✅ W1 COMPLETE — saved to w1_results.json")
print("=" * 70)
print(f"\n  Comparison context (existing 3-seed results):")
print(f"    M0 XLM-R baseline : 0.745 val macro-F1")
print(f"    M1 DarijaCharCNN  : 0.754 val macro-F1")
print(f"    M2 CharDarija     : 0.743 val macro-F1")
print(f"    W1 AraBERT (NEW)  : {vm['macro_f1']:.4f} val macro-F1")
print(f"\n  If W1 < 0.745: confirms XLM-R was the right choice for code-switched text")
print(f"  If W1 ≈ 0.745: AraBERT is competitive but not better, XLM-R still preferred")
print(f"  If W1 > 0.745: surprising; would need to rerun on 3 seeds before claiming")

# Restore XLM-R loaders so subsequent cells work normally
train_loader = _xlmr_train_loader
val_loader   = _xlmr_val_loader
test_loader  = _xlmr_test_loader
tokenizer = _xlmr_tokenizer
print(f"\n  ✓ XLM-R loaders restored")

del m_arabert
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── Retrain M2 seed 42 (lost the checkpoint, need it for gold test) ───
import gc, json, time
gc.collect(); torch.cuda.empty_cache()

print("=" * 70)
print("  RETRAIN — M2 seed 42")
print("=" * 70)

seed_everything(42)
m2 = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))

t0 = time.time()
m2, _, val_preds, val_labels = train_model(m2, "M2_CharGatedAdapter_seed42")
elapsed = (time.time() - t0) / 60
print(f"\n  Training time: {elapsed:.1f} min")

vm = compute_metrics(val_labels, val_preds, name="M2 seed42 VAL (retrain)")
_, _, te_preds, te_labels = _eval_epoch(m2, test_loader)
tm = compute_metrics(te_labels, te_preds, name="M2 seed42 TEST (retrain)")

# Save retrain results separately so we don't overwrite the original
m2_retrain = {
    "seed": 42,
    "val": vm,
    "test": tm,
    "training_minutes": elapsed,
    "checkpoint": f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed42.pt",
}
with open(f"{Config.SAVE_DIR}/m2_seed42_retrain.json", "w") as f:
    json.dump(m2_retrain, f, indent=2)

print(f"\n  ✓ Saved m2_seed42_retrain.json")
print(f"  ✓ Checkpoint at {m2_retrain['checkpoint']}")

del m2    
gc.collect(); torch.cuda.empty_cache()      

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Prepare 30-post stratified subsample for second annotator
# 10 mild + 10 moderate + 10 severe, sampled randomly within each class
# ─────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

np.random.seed(42)

LABEL_NAMES = {0: 'mild', 1: 'moderate', 2: 'severe'}

# Stratified sample: 10 per class from gold test
samples = []
for cls in [0, 1, 2]:
    pool = test_df[test_df['label'] == cls]
    n_pick = min(10, len(pool))
    picked = pool.sample(n=n_pick, random_state=42)
    samples.append(picked)
    print(f"  {LABEL_NAMES[cls]:>10}: {n_pick} posts from {len(pool)} available")

annotator_df = pd.concat(samples).reset_index(drop=True)

# Shuffle so annotator doesn't see them clustered by class
annotator_df = annotator_df.sample(frac=1, random_state=43).reset_index(drop=True)

# Strip out everything except what the annotator needs
out_df = pd.DataFrame({
    'id': annotator_df['id'],
    'text': annotator_df['text'],
    'your_label': '',
    'your_confidence': '',
    'your_notes': '',
})

# Save — annotator gets this version (no labels)
annotator_path = f"{Config.SAVE_DIR}/annotator_subset_BLIND.csv"
out_df.to_csv(annotator_path, index=False)

# Save — you keep this version (with your labels) for comparison
your_labels_df = pd.DataFrame({
    'id': annotator_df['id'],
    'text': annotator_df['text'],
    'oumaima_label': annotator_df['label'].map(LABEL_NAMES),
})
your_path = f"{Config.SAVE_DIR}/annotator_subset_OUMAIMA.csv"
your_labels_df.to_csv(your_path, index=False)

print(f"\n  ✓ Send to annotator: {annotator_path}")
print(f"  ✓ Keep private:      {your_path}")
print(f"\n  Total posts to label: {len(out_df)}")
print(f"  Class distribution: "
      f"mild={sum(annotator_df['label']==0)}, "
      f"moderate={sum(annotator_df['label']==1)}, "
      f"severe={sum(annotator_df['label']==2)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# Compute Cohen's kappa between your labels and second annotator's
# Run AFTER you receive the completed annotator CSV
# ─────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Load both files
mine     = pd.read_csv(f"{Config.SAVE_DIR}/annotator_subset_OUMAIMA.csv")
theirs   = pd.read_csv("/kaggle/working/annotator_subset_BLIND.csv")  # update this path

# Merge on id
merged = mine.merge(theirs[['id', 'your_label']], on='id', how='inner')
merged.rename(columns={'your_label': 'annotator2_label'}, inplace=True)

# Drop rows where annotator wrote "unclear" or left blank
valid = merged[merged['annotator2_label'].isin(['mild', 'moderate', 'severe'])]
n_valid = len(valid)
n_dropped = len(merged) - n_valid
print(f"  Valid annotations: {n_valid}, dropped: {n_dropped}")

# Compute kappa
y1 = valid['oumaima_label'].values
y2 = valid['annotator2_label'].values

kappa = cohen_kappa_score(y1, y2)
weighted_kappa = cohen_kappa_score(y1, y2, weights="quadratic")
agreement = (y1 == y2).mean()

print(f"\n  Cohen's κ (unweighted) : {kappa:.3f}")
print(f"  Cohen's κ (quadratic)  : {weighted_kappa:.3f}")
print(f"  Simple agreement       : {agreement:.1%}")

# Bootstrap CI for kappa
RNG = np.random.default_rng(42)
n = len(y1)
boot_kappa = np.empty(10000)
for i in range(10000):
    idx = RNG.integers(0, n, size=n)
    boot_kappa[i] = cohen_kappa_score(y1[idx], y2[idx])
lo, hi = np.percentile(boot_kappa, [2.5, 97.5])
print(f"  Kappa 95% CI           : [{lo:.3f}, {hi:.3f}]")

# Confusion matrix
print(f"\n  Confusion matrix (rows=Oumaima, cols=annotator):")
cm = confusion_matrix(y1, y2, labels=['mild', 'moderate', 'severe'])
print(f"  {'':10s} {'mild':>8s} {'moderate':>10s} {'severe':>8s}")
for i, row_label in enumerate(['mild', 'moderate', 'severe']):
    print(f"  {row_label:10s} {cm[i][0]:>8d} {cm[i][1]:>10d} {cm[i][2]:>8d}")

# List disagreements for adjudication
disagreements = valid[valid['oumaima_label'] != valid['annotator2_label']]
print(f"\n  Disagreements: {len(disagreements)} posts")
print(f"  (Review each and adjudicate by discussion)")

disagreements.to_csv(f"{Config.SAVE_DIR}/disagreements_for_adjudication.csv", 
                    index=False)
print(f"  ✓ Saved disagreements to: disagreements_for_adjudication.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os

labels = ['mild', 'moderate', 'severe']

plt.figure(figsize=(6.5, 5.5))
ax = sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels,
    cbar=True,
    linewidths=0.5,
    linecolor='white'
)

ax.set_xlabel('Annotator')
ax.set_ylabel('Oumaima')
ax.set_title("Confusion Matrix: Oumaima vs Second Annotator")

plt.tight_layout()

os.makedirs(Config.SAVE_DIR, exist_ok=True)
fig_path = f"{Config.SAVE_DIR}/confusion_matrix_annotators.png"
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved confusion matrix figure to: {fig_path}")

In [ ]:
from IPython.display import display
import ipywidgets as widgets

upload = widgets.FileUpload(
    accept='.csv',
    multiple=False
)

display(upload)

In [ ]:
import pandas as pd
import io

# get first uploaded file
file_info = upload.value[0]

# file name (this is the closest thing to a "path")
file_name = file_info['name']
print("File name:", file_name)

# Kaggle working directory path
file_path = "/kaggle/working/" + file_name
print("Kaggle path:", file_path)

# extract content
content = file_info['content']

# save file to disk (this creates the real path)
with open(file_path, "wb") as f:
    f.write(content)

# read csv from path
df = pd.read_csv(file_path)

df.head()

In [ ]:
import os
for name in ["M0_Baseline_seed42.pt",
             "M1_DarijaCharCNN_seed42.pt",
             "M2_CharGatedAdapter_seed42.pt",
             "W1_AraBERT_seed42.pt"]:
    path = f"{Config.SAVE_DIR}/{name}"
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f"  ✓ {name}  ({size:.0f} MB)")
    else:
        print(f"  ❌ {name}  MISSING")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# ALL-IN-ONE: retrain M0+M1+M2+AraBERT seed-42, then bootstrap CIs
# Total time: ~95 min compute + ~3 min bootstrap = ~100 min
# ─────────────────────────────────────────────────────────────────────
import os, gc, json, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel

# Sanity check before starting
for name, obj in [("CharCNN", "CharCNN"), ("BaselineXLMR", "BaselineXLMR"),
                  ("DarijaCharCNN", "DarijaCharCNN"),
                  ("CharGatedAdapterXLMR", "CharGatedAdapterXLMR"),
                  ("train_model", "train_model"),
                  ("compute_metrics", "compute_metrics"),
                  ("_eval_epoch", "_eval_epoch")]:
    if obj not in dir():
        raise RuntimeError(f"Missing: {obj}. Re-run earlier setup cells first.")
print("✓ All classes/functions loaded\n")
print(f"GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} / "
      f"{torch.cuda.mem_get_info()[1]/1e9:.1f} GB free\n")

ARABERT_NAME = "aubmindlab/bert-base-arabertv02"
T_START = time.time()

# ════════════════════════════════════════════════════════════════════
# STAGE 1 — M0 retrain
# ════════════════════════════════════════════════════════════════════
print("█" * 70)
print("  STAGE 1/4 — M0 seed 42")
print("█" * 70)
gc.collect(); torch.cuda.empty_cache()
seed_everything(42)
m0 = BaselineXLMR()
t0 = time.time()
m0, _, vp, vl = train_model(m0, "M0_Baseline_seed42")
print(f"\n  M0 training: {(time.time()-t0)/60:.1f} min")
m0_val = compute_metrics(vl, vp, name="M0 seed42 VAL")
_, _, tp, tl = _eval_epoch(m0, test_loader)
m0_test = compute_metrics(tl, tp, name="M0 seed42 TEST")
with open(f"{Config.SAVE_DIR}/m0_seed42_retrain.json", "w") as f:
    json.dump({"val": m0_val, "test": m0_test}, f, indent=2)
del m0; gc.collect(); torch.cuda.empty_cache()
print(f"  Elapsed total: {(time.time()-T_START)/60:.1f} min\n")

# ════════════════════════════════════════════════════════════════════
# STAGE 2 — M1 retrain
# ════════════════════════════════════════════════════════════════════
print("█" * 70)
print("  STAGE 2/4 — M1 seed 42")
print("█" * 70)
seed_everything(42)
m1 = DarijaCharCNN(char_vocab_size=len(char_vocab))
t0 = time.time()
m1, _, vp, vl = train_model(m1, "M1_DarijaCharCNN_seed42")
print(f"\n  M1 training: {(time.time()-t0)/60:.1f} min")
m1_val = compute_metrics(vl, vp, name="M1 seed42 VAL")
_, _, tp, tl = _eval_epoch(m1, test_loader)
m1_test = compute_metrics(tl, tp, name="M1 seed42 TEST")
with open(f"{Config.SAVE_DIR}/m1_seed42_retrain.json", "w") as f:
    json.dump({"val": m1_val, "test": m1_test}, f, indent=2)
del m1; gc.collect(); torch.cuda.empty_cache()
print(f"  Elapsed total: {(time.time()-T_START)/60:.1f} min\n")

# ════════════════════════════════════════════════════════════════════
# STAGE 3 — M2 retrain
# ════════════════════════════════════════════════════════════════════
print("█" * 70)
print("  STAGE 3/4 — M2 seed 42")
print("█" * 70)
seed_everything(42)
m2 = CharGatedAdapterXLMR(char_vocab_size=len(char_vocab))
t0 = time.time()
m2, _, vp, vl = train_model(m2, "M2_CharGatedAdapter_seed42")
print(f"\n  M2 training: {(time.time()-t0)/60:.1f} min")
m2_val = compute_metrics(vl, vp, name="M2 seed42 VAL")
_, _, tp, tl = _eval_epoch(m2, test_loader)
m2_test = compute_metrics(tl, tp, name="M2 seed42 TEST")
with open(f"{Config.SAVE_DIR}/m2_seed42_retrain.json", "w") as f:
    json.dump({"val": m2_val, "test": m2_test}, f, indent=2)
del m2; gc.collect(); torch.cuda.empty_cache()
print(f"  Elapsed total: {(time.time()-T_START)/60:.1f} min\n")

# ════════════════════════════════════════════════════════════════════
# STAGE 4 — AraBERT retrain (different tokenizer)
# ════════════════════════════════════════════════════════════════════
print("█" * 70)
print("  STAGE 4/4 — AraBERT seed 42")
print("█" * 70)

# Save XLM-R loaders, swap to AraBERT
_xlmr_train = train_loader
_xlmr_val = val_loader
_xlmr_test = test_loader
_xlmr_tok = tokenizer

ar_tok = AutoTokenizer.from_pretrained(ARABERT_NAME)
ar_train_ds = DarijaDataset(train_df, ar_tok, char_vocab)
ar_val_ds   = DarijaDataset(val_df,   ar_tok, char_vocab)
ar_test_ds  = DarijaDataset(test_df,  ar_tok, char_vocab)
ar_train_loader = DataLoader(ar_train_ds, batch_size=Config.BATCH_SIZE,
                             shuffle=True, num_workers=2, pin_memory=True)
ar_val_loader   = DataLoader(ar_val_ds, batch_size=Config.BATCH_SIZE*2,
                             shuffle=False, num_workers=2, pin_memory=True)
ar_test_loader  = DataLoader(ar_test_ds, batch_size=Config.BATCH_SIZE*2,
                             shuffle=False, num_workers=2, pin_memory=True)

train_loader = ar_train_loader
val_loader = ar_val_loader
test_loader = ar_test_loader

class AraBERTBaseline(nn.Module):
    def __init__(self, model_name=ARABERT_NAME, num_classes=Config.NUM_CLASSES, **kw):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        d = self.xlmr.config.hidden_size
        self.classifier = nn.Sequential(nn.Dropout(0.3), CoralLayer(d, num_classes))
    def forward(self, input_ids, attention_mask, **kwargs):
        cls = self.xlmr(input_ids=input_ids,
                        attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(cls)

seed_everything(42)
m_ar = AraBERTBaseline()
t0 = time.time()
m_ar, _, vp, vl = train_model(m_ar, "W1_AraBERT_seed42")
print(f"\n  AraBERT training: {(time.time()-t0)/60:.1f} min")
ar_val = compute_metrics(vl, vp, name="AraBERT seed42 VAL")
_, _, tp, tl = _eval_epoch(m_ar, test_loader)
ar_test = compute_metrics(tl, tp, name="AraBERT seed42 TEST")
with open(f"{Config.SAVE_DIR}/w1_results.json", "w") as f:
    json.dump({"model": ARABERT_NAME, "val": ar_val, "test": ar_test}, f, indent=2)
del m_ar; gc.collect(); torch.cuda.empty_cache()

# Restore XLM-R loaders BEFORE bootstrap (M0/M1/M2 need them)
train_loader = _xlmr_train
val_loader = _xlmr_val
test_loader = _xlmr_test
tokenizer = _xlmr_tok
print(f"\n  ✓ XLM-R loaders restored")
print(f"  Elapsed total: {(time.time()-T_START)/60:.1f} min\n")

# ════════════════════════════════════════════════════════════════════
# STAGE 5 — Bootstrap CIs (uses all 4 checkpoints we just saved)
# ════════════════════════════════════════════════════════════════════
print("█" * 70)
print("  BOOTSTRAP CI ANALYSIS")
print("█" * 70)

predictions = {}
true_labels = None

@torch.no_grad()
def _predict(ckpt_path, ModelClass, model_kwargs, loader, needs_chars):
    seed_everything(42)
    m = ModelClass(**model_kwargs).to(Config.DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=Config.DEVICE))
    if hasattr(m, '_char_active'):
        m._char_active = True
    m.eval()
    preds, labs = [], []
    for batch in loader:
        ids   = batch["input_ids"].to(Config.DEVICE)
        mask  = batch["attention_mask"].to(Config.DEVICE)
        lbls  = batch["label"]
        if needs_chars:
            chars = batch["char_ids"].to(Config.DEVICE)
            logits = m(input_ids=ids, attention_mask=mask, char_ids=chars)
        else:
            logits = m(input_ids=ids, attention_mask=mask)
        preds.extend(corn_label_from_logits(logits).cpu().tolist())
        labs.extend(lbls.tolist())
    del m; gc.collect(); torch.cuda.empty_cache()
    return np.array(preds), np.array(labs)


# M0 — XLM-R, no chars
print("\n  Loading M0 predictions...")
p, l = _predict(f"{Config.SAVE_DIR}/M0_Baseline_seed42.pt",
                BaselineXLMR, {}, test_loader, needs_chars=False)
predictions['M0'] = p; true_labels = l

# M1 — XLM-R, with chars
print("  Loading M1 predictions...")
p, _ = _predict(f"{Config.SAVE_DIR}/M1_DarijaCharCNN_seed42.pt",
                DarijaCharCNN, {"char_vocab_size": len(char_vocab)},
                test_loader, needs_chars=True)
predictions['M1'] = p

# M2 — XLM-R, with chars
print("  Loading M2 predictions...")
p, _ = _predict(f"{Config.SAVE_DIR}/M2_CharGatedAdapter_seed42.pt",
                CharGatedAdapterXLMR, {"char_vocab_size": len(char_vocab)},
                test_loader, needs_chars=True)
predictions['M2'] = p

# AraBERT — needs its own loader
print("  Loading AraBERT predictions...")
m_ar = AraBERTBaseline().to(Config.DEVICE)
m_ar.load_state_dict(torch.load(f"{Config.SAVE_DIR}/W1_AraBERT_seed42.pt",
                                  map_location=Config.DEVICE))
m_ar.eval()
preds = []
with torch.no_grad():
    for batch in ar_test_loader:
        ids = batch["input_ids"].to(Config.DEVICE)
        mask = batch["attention_mask"].to(Config.DEVICE)
        logits = m_ar(input_ids=ids, attention_mask=mask)
        preds.extend(corn_label_from_logits(logits).cpu().tolist())
predictions['AraBERT'] = np.array(preds)
del m_ar; gc.collect(); torch.cuda.empty_cache()

print(f"\n  ✓ All 4 models scored on {len(true_labels)} test posts\n")

# Bootstrap
N_BOOT = 10000
RNG = np.random.default_rng(42)
n = len(true_labels)

def macro_f1(t, p): return f1_score(t, p, average="macro", zero_division=0)
def severe_f1(t, p): return f1_score(t, p, average=None, zero_division=0, labels=[0,1,2])[2]

def boot_single(fn, preds):
    pt = fn(true_labels, preds)
    b = np.empty(N_BOOT)
    for i in range(N_BOOT):
        idx = RNG.integers(0, n, size=n)
        b[i] = fn(true_labels[idx], preds[idx])
    return pt, np.percentile(b, 2.5), np.percentile(b, 97.5)

def boot_paired(fn, a, b):
    pt = fn(true_labels, a) - fn(true_labels, b)
    out = np.empty(N_BOOT)
    for i in range(N_BOOT):
        idx = RNG.integers(0, n, size=n)
        out[i] = fn(true_labels[idx], a[idx]) - fn(true_labels[idx], b[idx])
    return pt, np.percentile(out, 2.5), np.percentile(out, 97.5)

print("─" * 70)
print(f"  Per-model 95% bootstrap CIs (10,000 resamples)")
print("─" * 70)
print(f"  {'Model':<10} {'Macro-F1 [95% CI]':<28} {'Severe-F1 [95% CI]':<28}")
print("  " + "─" * 66)

single = {}
for m in ['M0', 'M1', 'M2', 'AraBERT']:
    f, fl, fh = boot_single(macro_f1, predictions[m])
    s, sl, sh = boot_single(severe_f1, predictions[m])
    print(f"  {m:<10} {f:.3f} [{fl:.3f}, {fh:.3f}]      {s:.3f} [{sl:.3f}, {sh:.3f}]")
    single[m] = {'macro_f1': [float(f), float(fl), float(fh)],
                 'severe_f1': [float(s), float(sl), float(sh)]}

print("\n" + "─" * 70)
print(f"  Paired bootstrap CIs (Δ macro-F1, same resamples)")
print("─" * 70)
print(f"  {'Comparison':<20} {'Δ macro-F1 [95% CI]':<28} {'Verdict':<20}")
print("  " + "─" * 66)

paired = {}
for a, b in [('M1','M0'), ('M2','M0'), ('M2','M1'), ('AraBERT','M0')]:
    d, dl, dh = boot_paired(macro_f1, predictions[a], predictions[b])
    sig = (dl > 0) or (dh < 0)
    verdict = "significant ✓" if sig else "directional"
    print(f"  {a+' vs '+b:<20} {d:+.3f} [{dl:+.3f}, {dh:+.3f}]     {verdict}")
    paired[f"{a}_vs_{b}"] = {'delta': float(d), 'lo': float(dl), 'hi': float(dh),
                              'significant': bool(sig)}

with open(f"{Config.SAVE_DIR}/bootstrap_ci_results.json", 'w') as f:
    json.dump({'n_test': int(n), 'n_bootstrap': N_BOOT,
               'per_model': single, 'paired': paired}, f, indent=2)

total = (time.time() - T_START) / 60
print("\n" + "█" * 70)
print(f"  ✅ ALL DONE in {total:.1f} minutes")
print("█" * 70)
print(f"\n  ⚠ NOW DO:")
print(f"     1. File → Save Version → Quick Save (PERSIST CHECKPOINTS)")
print(f"     2. Download these files to laptop:")
print(f"        - bootstrap_ci_results.json")
print(f"        - m0_seed42_retrain.json")
print(f"        - m1_seed42_retrain.json")
print(f"        - m2_seed42_retrain.json")
print(f"        - w1_results.json")
print(f"     3. Optionally download the 4 .pt checkpoints (~4 GB)")          